# Solasterid v4 — Async Wave Scheduler + Engine-Aware Evolution

This is the next iteration, written **from Solasterid v3's own v4 request** plus a
direct reading of what actually went wrong in the v3 run logs (`run_20260524_124945_321850e4`).

## What v3 asked for, and what was really broken

Solasterid asked for a *"lease-based async task graph,"* a *"spawn broker,"* and
*"loop-break escalation,"* and it tried to get them by **adding arms** (`Async Scheduler`,
`Dynamic Specialist Spawner`, `Spawn Broker`, `Async Orchestrator`). That could never work,
and the logs prove why:

| Symptom in the v3 run | Real cause |
|---|---|
| 48 active arms, 42 committees, but only **7 arms / 4 committees ever spoke** | The scheduler popped the **first 4** committees of a fixed, layer-sorted queue (`MAX_COMMITTEES_PER_STEP = 4`) and almost always `FINAL_RENDER`ed on step 1. |
| Newly spawned specialists did nothing | They were auto-wired into committees that **are never visited**, so spawning was invisible. |
| "Schema loop" / endless self-questioning | The same 4 governance/meta committees met every run, so the system only ever talked about its own schema. Domain committees were unreachable. |
| Adding a "scheduler arm" never helped | **An arm is just a prompt.** Scheduling lives in the orchestrator *code*, which the system cannot mutate. It kept prescribing medicine it had no way to swallow. |

**v4's thesis:** the bottleneck was never reasoning depth or arm count. It was the
**orchestrator topology**. So v4 fixes the engine, not the roster.

## What v4 adds (engine-level — things v3 could not self-mutate)

1. **Wave scheduler (`run_wave_schedule`).** Replaces the front-popping queue. Committees
   are scheduled in **dependency waves** by layer; within a wave they are independent and
   visited up to a real budget (`MAX_COMMITTEES_PER_WAVE`, default 12). Coverage is tracked:
   the loop guarantees every *relevant* committee gets a turn before synthesis, instead of
   the same 4 forever.
2. **Coverage-first termination.** `FINAL_RENDER` on step 1 is allowed **only after a
   minimum committee/arm coverage fraction is met** (`MIN_COMMITTEE_COVERAGE`), so the system
   can't shortcut to a meta-answer before its specialists have spoken.
3. **Real dynamic specialist spawning (`maybe_spawn_specialist`), in code.** When a committee
   reports a `domain_gap`, the orchestrator *itself* mints a new specialist arm mid-run and
   routes the live subtask to it — the thing the "Spawn Broker arm" was pretending to do.
4. **Loop-break detector (`detect_schema_loop`), in code.** If the same falsifier signature /
   schema request repeats across waves, the orchestrator forcibly reroutes to an
   unvisited domain committee instead of re-deliberating the same meta-point.
5. **Engine-aware Evolutionbot.** It is now told, explicitly, that scheduling/spawning/looping
   are **code organs, not arms**, and is forbidden from proposing arms whose lens is a control
   mechanism. Its job shifts to what arms *can* do: domain expertise and committee topology.
6. **Branch-local scratch.** Each committee wave writes to an isolated scratch namespace so
   parallel committees can't overwrite each other's findings before the merge.

**Everything v3 already got right is preserved unchanged:** the code-enforced liveness FSM,
deterministic `matched`, the load-time sanitizers + quarantine/known-good revert, transactional
"apply-valid-drop-bad" mutations, per-arm web access, seed-memory hygiene, and unique run dirs.

No API key is hardcoded. Set `OPENAI_API_KEY` in your environment.


In [ ]:
# Install if needed:
# %pip install openai rich

import os, json, time, uuid, hashlib, re, copy, textwrap
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from collections import Counter, defaultdict

try:
    from openai import OpenAI
except Exception as e:
    raise RuntimeError("Install the OpenAI SDK first: pip install openai") from e

try:
    from rich import print as rprint
except Exception:
    rprint = print

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

# A connect timeout keeps a stalled network read from hanging the whole run.
client = OpenAI(timeout=60.0, max_retries=2)
rprint("[green]SDK client ready. Key loaded without printing it.[/green]")


In [ ]:
# =========================
# Global configuration knobs
# =========================

STORAGE_ROOT = Path("solasterid_v4")
RUNS_DIR = STORAGE_ROOT / "runs"
ARCH_DIR = STORAGE_ROOT / "architectures"
MEMORY_DIR = STORAGE_ROOT / "memory"
LOG_DIR = STORAGE_ROOT / "logs"

for p in [STORAGE_ROOT, RUNS_DIR, ARCH_DIR, MEMORY_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Models are variables on purpose. Swap depending on availability / cost.
LISTENER_MODEL = "gpt-5.4-mini"
ARM_MODEL = "gpt-5.4-mini"
SPEAKER_MODEL = "gpt-5.4-mini"
EVOLUTION_MODEL = "gpt-5.4-mini"

FALLBACK_MODEL = "gpt-5.4-mini"

# Toggle web access for arm/listener/speaker/evolution calls. The tool name is the
# current Responses API tool ("web_search"); the v0.4 typo "websearch" silently failed.
ENABLE_WEB_SEARCH = True            # Master switch. If False, nobody searches.
WEB_SEARCH_TOOL = {"type": "web_search"}

# Per-role web access. Each arm ALSO carries its own "web_access" flag (see DEFAULT_ARMS),
# so arms search independently of the listener/speaker/evolution roles.
LISTENER_WEB_ACCESS = False         # The listener only paraphrases; it doesn't need to search.
SPEAKER_WEB_ACCESS = True           # The speaker may verify before an atomic final render.
EVOLUTION_WEB_ACCESS = True         # Evolutionbot may research architecture ideas.
DEFAULT_ARM_WEB_ACCESS = True       # Default for newly-created arms.

DEFAULT_GLOBALS = {
    # Core liveness
    "MAX_OUTPUT_STEPS": 6,                   # Speaker gets at most this many chances.
    "MAX_DELIBERATION_ROUNDS_PER_STEP": 2,   # Board passes before a speaker decision.
    "MIN_DELIBERATION_STEPS": 1,
    "MAX_TOTAL_API_CALLS": 800,              # Hard ceiling; auto-scaled at run start if roster outgrows it.

    # Atomic rendering
    "ATOMIC_FINAL_RENDER": True,
    "ALLOW_PARTIAL_VISIBLE_STREAM": False,
    "FINAL_RENDER_WORD_LIMIT": 4200,
    "EXACT_FORMAT_HARD_GATE": True,
    "FINAL_SELF_CHECK_REQUIRED": True,

    # v0.5 liveness gate
    "LIVENESS_GATE_ENABLED": True,           # Code-enforced; blocks FINAL_RENDER on failure.
    "LIVENESS_GATE_REQUIRED_FIELDS": [
        "speaker_msg_id", "speaker_timestamp", "present", "matched", "missing_reason",
    ],

    # Deliberation pressure
    "EARLY_FINAL_IF_MEAN_CONFIDENCE_GTE": 0.97,   # v4: raise the early-exit bar.
    "CONTINUE_PENALTY_AFTER_STEP": 3,
    "MAX_CONTINUE_ACTIONS": 4,

    # Context windows (kept modest so prompts don't bloat with a large arm roster)
    "SHARED_MEMORY_TOP_K": 8,
    "PERSONAL_MEMORY_TOP_K": 5,
    "SHARED_SCRATCH_RECENT_K": 30,
    "LOCAL_CONTEXT_RECENT_K": 30,

    # Seed memory hygiene
    "MAX_SEED_MEMORIES": 8,                  # Hard cap; oldest non-core seeds are compacted.

    # Printing
    "PRINT_FULL_ARM_REPORTS": False,
    "PRINT_SPEAKER_DECISIONS": True,

    # Evolution
    "ALLOW_MUTATIONS": True,
    "MAX_ARM_COUNT": 128,                    # Room to grow toward the user's goal.
    "MIN_ACTIVE_ARMS": 5,                   # Floor: retirement blocked below this.
    "MAX_NEW_ARMS_PER_EVOLUTION": 10,         # Allow meaningful growth each cycle.

    # ---- v4: WAVE SCHEDULER (the actual fix for "only 4 committees ever met") ----
    # v3 popped the first MAX_COMMITTEES_PER_STEP=4 committees of a fixed queue and
    # FINAL_RENDERed on step 1, so 41/48 arms never spoke. v4 schedules committees in
    # dependency *waves* by layer; within a wave they are independent and we visit up to
    # MAX_COMMITTEES_PER_WAVE of them. Coverage is tracked across waves.
    "USE_WAVE_SCHEDULER": True,               # Master switch for the v4 orchestrator.
    "MAX_COMMITTEES_PER_WAVE": 12,            # How many committees may meet in one wave.
    "MAX_WAVES_PER_STEP": 4,                  # Dependency depth (layers) explored per step.
    "MAX_COMMITTEE_VISITS_TOTAL": 60,         # Hard ceiling on committee meetings per run.

    # ---- v4: COVERAGE-FIRST TERMINATION ----
    # Block an early FINAL_RENDER until enough of the *relevant* committees/arms have
    # actually contributed. Stops the system shortcutting to a meta-answer about itself.
    "MIN_COMMITTEE_COVERAGE": 0.45,           # Fraction of scheduled committees that must report.
    "MIN_DISTINCT_ARMS_BEFORE_FINAL": 10,     # At least this many distinct arms must have spoken.
    "COVERAGE_GATE_ENABLED": True,

    # ---- v4: DYNAMIC SPECIALIST SPAWNING (in code, not as an arm) ----
    # When a committee reports a domain_gap, the ORCHESTRATOR mints a specialist arm and
    # routes the live subtask to it. This is what the "Spawn Broker arm" could never do.
    "DYNAMIC_SPAWN_ENABLED": True,
    "MAX_DYNAMIC_SPAWNS_PER_RUN": 4,          # Cap in-run organ growth.
    "DYNAMIC_SPAWN_MIN_CONFIDENCE": 0.55,     # A gap claim below this is ignored as noise.

    # ---- v4: LOOP-BREAK DETECTOR (in code) ----
    # If the same falsifier/schema signature repeats across waves, reroute to an unvisited
    # domain committee instead of re-litigating the same meta-point ("the exact schema" loop).
    "LOOP_BREAK_ENABLED": True,
    "LOOP_BREAK_REPEAT_THRESHOLD": 2,         # Same signature this many times => break.

    # ---- v4: BRANCH-LOCAL SCRATCH ----
    # Each committee wave writes to an isolated scratch namespace so parallel committees
    # cannot clobber each other's findings before the merge step.
    "BRANCH_LOCAL_SCRATCH": True,
}

LISTENER_TEMPERATURE = 0.50
EVOLUTION_TEMPERATURE = 0.20

# Default arms. These become persistent in architecture_state.json.
DEFAULT_ARMS = {
    "arm_1": {
        "name": "Literalist",
        "lens": "Exact wording, scope, definitions, formatting constraints, and what the user literally asked.",
        "generation": 0, "parents": [], "retired": False, "status": "active", "web_access": True,
        "tuning": {"temperature": 0.25, "doubt": 0.55, "creativity": 0.20, "precision": 0.92,
                   "shared_memory_weight": 0.85, "personal_memory_weight": 0.55, "local_context_weight": 0.65},
    },
    "arm_2": {
        "name": "Historian",
        "lens": "Continuity, prior context, remembered project logic, persistent user intent, and avoiding commitment churn.",
        "generation": 0, "parents": [], "retired": False, "status": "active", "web_access": True,
        "tuning": {"temperature": 0.40, "doubt": 0.45, "creativity": 0.45, "precision": 0.75,
                   "shared_memory_weight": 0.95, "personal_memory_weight": 0.80, "local_context_weight": 0.70},
    },
    "arm_3": {
        "name": "Mechanic",
        "lens": "Implementation feasibility, architecture, runtime, code shape, state machines, and failure states.",
        "generation": 0, "parents": [], "retired": False, "status": "active", "web_access": True,
        "tuning": {"temperature": 0.30, "doubt": 0.65, "creativity": 0.35, "precision": 0.88,
                   "shared_memory_weight": 0.80, "personal_memory_weight": 0.70, "local_context_weight": 0.75},
    },
    "arm_4": {
        "name": "Adversary",
        "lens": "Contradictions, overclaims, safety problems, epistemic risk, false consensus, and missing verification.",
        "generation": 0, "parents": [], "retired": False, "status": "active", "web_access": True,
        "tuning": {"temperature": 0.35, "doubt": 0.95, "creativity": 0.30, "precision": 0.90,
                   "shared_memory_weight": 0.75, "personal_memory_weight": 0.85, "local_context_weight": 0.70},
    },
    "arm_5": {
        "name": "Dreamer",
        "lens": "Metaphor, emotional resonance, generativity, aesthetic coherence, and living language without replacing mechanics.",
        "generation": 0, "parents": [], "retired": False, "status": "active", "web_access": True,
        "tuning": {"temperature": 0.70, "doubt": 0.35, "creativity": 0.90, "precision": 0.55,
                   "shared_memory_weight": 0.70, "personal_memory_weight": 0.80, "local_context_weight": 0.85},
    },
}


# Default committee scaffold. This DOES NOT change your existing model/temperature/run
# parameters; it just gives the architecture a first-class place to store groups.
DEFAULT_COMMITTEES = {
    "core_reasoning": {
        "name": "Core Reasoning Council",
        "layer": 0,
        "leader": "arm_1",
        "members": ["arm_1", "arm_3", "arm_4", "arm_5"],
        "parent": None,
        "children": ["semantic_detectors", "domain_specialists"],
        "routing_hint": "First-pass interpretation, feasibility, objections, and generative candidate answers.",
        "output_policy": "May release findings directly to Speakerbot or route subtasks to specialist committees.",
        "status": "active",
    },
    "governance_safety": {
        "name": "Governance and Safety Committee",
        "layer": 1,
        "leader": "arm_4",
        "members": ["arm_4", "arm_1"],
        "parent": None,
        "children": ["evolution_experimentation"],
        "routing_hint": "Use for constraints, scope, safety, overclaim risk, schema pressure, and authority boundaries.",
        "output_policy": "Usually reviews before final synthesis when risk or self-modification is involved.",
        "status": "active",
    },
    "memory_replay": {
        "name": "Memory and Replay Committee",
        "layer": 1,
        "leader": "arm_1",
        "members": ["arm_1"],
        "parent": None,
        "children": [],
        "routing_hint": "Use for continuity, prior runs, replay checks, and whether findings survive rerun logic.",
        "output_policy": "Releases only compact continuity and replay findings.",
        "status": "active",
    },
    "evolution_experimentation": {
        "name": "Evolution and Experimentation Committee",
        "layer": 2,
        "leader": "arm_3",
        "members": ["arm_3", "arm_4"],
        "parent": "governance_safety",
        "children": [],
        "routing_hint": "Use for architecture mutations, rollback, benchmarks, incentives, and whether new organs are worth their metabolic cost.",
        "output_policy": "Must be concrete: proposed mutation, expected effect, rollback condition.",
        "status": "active",
    },
    "semantic_detectors": {
        "name": "Semantic Detectors",
        "layer": 1,
        "leader": "arm_1",
        "members": ["arm_1", "arm_5"],
        "parent": "core_reasoning",
        "children": [],
        "routing_hint": "Use for hidden assumptions, missing abstraction layers, metaphor/meaning drift, and terms that carry unstated ontologies.",
        "output_policy": "Releases short semantic risk notes and better framings.",
        "status": "active",
    },
    "domain_specialists": {
        "name": "Domain Specialists",
        "layer": 1,
        "leader": "arm_3",
        "members": ["arm_3"],
        "parent": "core_reasoning",
        "children": [],
        "routing_hint": "Use when math, science, medicine, code, law-like constraints, or current external facts need jurisdiction.",
        "output_policy": "Routes to the most relevant specialist arms and marks uncertainty explicitly.",
        "status": "active",
    },
    "synthesis_cabinet": {
        "name": "Synthesis Cabinet",
        "layer": 3,
        "leader": "arm_1",
        "members": ["arm_1", "arm_3", "arm_4", "arm_5"],
        "parent": None,
        "children": [],
        "routing_hint": "Late-stage compression, reconciliation, and final-answer readiness.",
        "output_policy": "Should run near the end when multiple committees produced findings.",
        "status": "active",
    },
}

ARCH_STATE_PATH = STORAGE_ROOT / "architecture_state.json"
MUTATION_LOG_PATH = LOG_DIR / "mutation_log.jsonl"
SHARED_MEMORY_PATH = MEMORY_DIR / "shared_memory.jsonl"

rprint("[green]Config ready (v4: wave scheduler + coverage gate + in-code spawning).[/green]")


In [ ]:
# =========================
# Persistence helpers
# =========================

def now_iso():
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")

def short_id(prefix="id"):
    return f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

def sha256_text(x):
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:16]

def atomic_write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex[:8]}")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def read_jsonl(path, max_items=None):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows[-max_items:] if max_items else rows

def make_unique_run_dir(run_id=None):
    run_id = run_id or short_id("run")
    run_dir = RUNS_DIR / run_id
    if run_dir.exists():
        run_dir = RUNS_DIR / f"{run_id}_{uuid.uuid4().hex[:6]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_id, run_dir

def save_run_file(run_dir, name, obj):
    path = Path(run_dir) / name
    atomic_write_json(path, obj)
    return path

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default

def sanitize_text_for_api(s):
    """Force text to pure ASCII so no non-ASCII character can crash an outbound API call.
    The 'ascii codec can't encode \u201c' crash came from smart quotes / dashes in prior
    bot text being re-sent into a new API call. We first replace common typographic
    characters with their ASCII equivalents, then hard-encode the entire string to ASCII
    with replacement so NOTHING non-ASCII survives."""
    if s is None:
        return ""
    if not isinstance(s, str):
        try:
            s = json.dumps(s, ensure_ascii=True)
        except Exception:
            s = str(s)
    replacements = {
        "\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'",
        "\u2013": "-", "\u2014": "--", "\u2026": "...", "\u00a0": " ",
        "\u2212": "-", "\u00ab": '"', "\u00bb": '"', "\u2032": "'", "\u2033": '"',
        "\u2192": "->", "\u2190": "<-", "\u2194": "<->", "\u2022": "*",
        "\u00b7": ".", "\u00e9": "e", "\u00e8": "e",
        "\u00e0": "a", "\u00fc": "u", "\u00f1": "n", "\u00e7": "c",
    }
    for bad, good in replacements.items():
        s = s.replace(bad, good)
    # Hard gate: force to pure ASCII. Any remaining non-ASCII char becomes '?'.
    return s.encode("ascii", "replace").decode("ascii")

def force_ascii_deep(obj):
    """Recursively walk a dict/list and force every string value to pure ASCII.
    Applied to all API outputs before they enter the transcript/scratch/memory,
    preventing non-ASCII bot output from poisoning the next call's input."""
    if isinstance(obj, str):
        return sanitize_text_for_api(obj)
    if isinstance(obj, dict):
        return {force_ascii_deep(k): force_ascii_deep(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [force_ascii_deep(item) for item in obj]
    return obj

def is_error_obj(obj):
    """True if obj is a wrapper error/parse-error dict, not a real model result.
    Downstream code checks this before treating obj as a normal report/decision, so an
    error dict can never be silently consumed (the source of the dict-attr cascade)."""
    return isinstance(obj, dict) and bool(
        obj.get("_error") or obj.get("_parse_error") or obj.get("_fallback_error"))

def extract_json_object(text):
    """Best-effort JSON extraction from model text."""
    if isinstance(text, dict):
        return text
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{"); end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    start = text.find("["); end = text.rfind("]")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    return {"_parse_error": True, "_raw": text}

def tokenish_truncate(text, max_chars=3000):
    text = "" if text is None else str(text)
    return text if len(text) <= max_chars else text[:max_chars] + "\n...[truncated]"

# ---- v0.5 typing/coercion helpers (notes item 4) ----

IMPORTANCE_LABELS = {"high": 0.9, "medium": 0.7, "med": 0.7, "low": 0.5, "critical": 0.95, "p0": 0.95}

def coerce_importance(value, default=0.75):
    """Accept floats, numeric strings, or labels like high/medium/low.
    Never raises; returns a float in [0, 1]."""
    if isinstance(value, (int, float)):
        v = float(value)
    elif isinstance(value, str):
        s = value.strip().lower()
        if s in IMPORTANCE_LABELS:
            v = IMPORTANCE_LABELS[s]
        else:
            try:
                v = float(s)
            except ValueError:
                v = default
    else:
        v = default
    return max(0.0, min(1.0, v))

rprint("[green]Persistence helpers ready.[/green]")



In [ ]:
# =========================
# Architecture state + built-in sanitizer
# =========================

CANONICAL_TUNING = {
    "temperature": 0.40, "doubt": 0.55, "creativity": 0.35, "precision": 0.75,
    "shared_memory_weight": 0.75, "personal_memory_weight": 0.75, "local_context_weight": 0.70,
}

PLACEHOLDER_LENS_PATTERNS = [
    r"^\s*$",
    r"A new specialized lens proposed by Evolutionbot\.?",
    r"new specialized lens",
    r"proposed by Evolutionbot",
]

def is_placeholder_lens(lens):
    if not isinstance(lens, str):
        return True
    return any(re.search(p, lens, flags=re.I) for p in PLACEHOLDER_LENS_PATTERNS)

def sanitize_arms(state):
    """Repair the exact corruption modes seen in v0.4 runs (notes item 5):
      - name is a dict (add_arm spec leaked into the name field)
      - lens is a placeholder / blank / non-string
      - name is an entire sentence (lens text leaked into name)
      - retired/status disagree
    Returns (state, repairs)."""
    repairs = []
    arms = state.get("arms", {})
    for arm_id, arm in arms.items():
        if not isinstance(arm, dict):
            continue

        # 1) Name is a dict -> it's actually a nested arm spec. Pull fields out.
        name = arm.get("name")
        if isinstance(name, dict):
            spec = name
            arm["name"] = spec.get("name") if isinstance(spec.get("name"), str) else f"Arm_{arm_id}"
            if is_placeholder_lens(arm.get("lens")) and isinstance(spec.get("lens"), str):
                arm["lens"] = spec["lens"]
            for k in ("generation", "parents", "tuning"):
                if k in spec and (k not in arm or not arm.get(k)):
                    arm[k] = spec[k]
            repairs.append((arm_id, "name_dict_unpacked", arm["name"]))
            name = arm["name"]

        # 2) Name missing / non-string
        if not isinstance(arm.get("name"), str) or not arm["name"].strip():
            arm["name"] = f"Arm_{arm_id}"
            repairs.append((arm_id, "name_filled", arm["name"]))
            name = arm["name"]

        # 3) Name is a whole sentence -> derive a short handle, move text to lens if needed.
        if len(name) > 48 or name.count(" ") > 6:
            if is_placeholder_lens(arm.get("lens")):
                arm["lens"] = name if not name.endswith(".") else name
            short = re.sub(r"[^A-Za-z0-9]+", "", name.split(":")[0].split(".")[0].title())[:32] or f"Arm_{arm_id}"
            arm["name"] = short
            repairs.append((arm_id, "long_name_shortened", short))

        # 4) Placeholder/blank lens -> synthesize from name.
        if is_placeholder_lens(arm.get("lens")):
            arm["lens"] = (f"Inspect the task through the {arm['name']} function; "
                           "produce concrete, schema-valid reports without redundant role sprawl.")
            repairs.append((arm_id, "lens_synthesized", arm["name"]))

        # 5) Structural defaults
        arm.setdefault("generation", 0)
        arm.setdefault("parents", [])
        arm.setdefault("web_access", DEFAULT_ARM_WEB_ACCESS)
        tuning = arm.setdefault("tuning", {})
        for k, v in CANONICAL_TUNING.items():
            tuning.setdefault(k, v)

        # 6) retired/status consistency
        retired = bool(arm.get("retired", False)) or arm.get("status") == "retired"
        arm["retired"] = retired
        arm["status"] = "retired" if retired else "active"

    return state, repairs


# =========================
# Committee substrate
# =========================
#
# Committees are metadata/routing organs layered over arms. They do not replace arms:
# - one arm can sit on multiple committees;
# - committees can be nested;
# - leaders translate broad committee tasks into member-specific prompts;
# - the run loop can route/bounce committee findings before Speakerbot sees them.

COMMITTEE_NAME_KEYWORDS = {
    "core_reasoning": [
        "literalist", "mechanic", "adversary", "dreamer",
    ],
    "governance_safety": [
        "adversary", "schema", "sentinel", "assumption", "boundary", "protocol",
        "incentive", "regime",
    ],
    "memory_replay": [
        "memory", "replay", "historian", "signal steward",
    ],
    "evolution_experimentation": [
        "cycle", "growth", "rollback", "benchmark", "incentive", "regime",
        "protocol economist", "mechanic",
    ],
    "semantic_detectors": [
        "absent-layer", "absent layer", "semantics", "literalist", "dreamer",
    ],
    "domain_specialists": [
        "math", "science", "medicine", "computer-science", "computer science",
        "specialist", "sage", "cartographer",
    ],
    "synthesis_cabinet": [
        "literalist", "mechanic", "adversary", "dreamer", "signal steward",
    ],
}

def _active_existing_arm_ids(state):
    arms = state.get("arms", {})
    return [aid for aid, arm in arms.items()
            if isinstance(arm, dict)
            and not arm.get("retired", False)
            and arm.get("status", "active") != "retired"]

def _find_arm_ids_by_keywords(state, keywords):
    out = []
    for aid in _active_existing_arm_ids(state):
        arm = state["arms"][aid]
        hay = f"{arm.get('name','')} {arm.get('lens','')}".lower()
        if any(str(k).lower() in hay for k in keywords):
            out.append(aid)
    return out

def _first_existing(state, ids, fallback=None):
    active = set(_active_existing_arm_ids(state))
    for aid in ids or []:
        if aid in active:
            return aid
    if fallback in active:
        return fallback
    return next(iter(active), None)

def build_default_committees_for_state(state):
    """Derive a sane committee scaffold from whatever arms currently exist.
    This lets an evolved architecture with arm_27 etc. get committee membership
    without hard-coding those arms into DEFAULT_COMMITTEES."""
    active = _active_existing_arm_ids(state)
    if not active:
        return {}
    committees = copy.deepcopy(DEFAULT_COMMITTEES)
    for cid, committee in committees.items():
        keyword_members = _find_arm_ids_by_keywords(state, COMMITTEE_NAME_KEYWORDS.get(cid, []))
        seed_members = [aid for aid in committee.get("members", []) if aid in active]
        members = []
        for aid in seed_members + keyword_members:
            if aid in active and aid not in members:
                members.append(aid)
        if not members:
            members = active[:min(4, len(active))]
        leader = _first_existing(state, [committee.get("leader")] + members, fallback=members[0])
        committee["leader"] = leader
        committee["members"] = members
        committee.setdefault("status", "active")
        committee.setdefault("layer", 0)
        committee.setdefault("children", [])
        committee.setdefault("parent", None)
        committee.setdefault("routing_hint", "")
        committee.setdefault("output_policy", "Release compact findings to the next routing layer or Speakerbot.")
    # Drop parent/child references that do not exist after normalization.
    cids = set(committees)
    for cid, c in committees.items():
        c["children"] = [x for x in c.get("children", []) if x in cids and x != cid]
        if c.get("parent") not in cids:
            c["parent"] = None
    return committees

def sanitize_committees(state):
    """Normalize committee metadata and auto-create it when missing.
    Returns (state, repairs)."""
    repairs = []
    active = set(_active_existing_arm_ids(state))
    if not isinstance(state.get("committees"), dict) or not state.get("committees"):
        state["committees"] = build_default_committees_for_state(state)
        repairs.append(("_committees", "created_default_committee_scaffold", len(state["committees"])))

    committees = state.get("committees", {})
    if not isinstance(committees, dict):
        state["committees"] = build_default_committees_for_state(state)
        return state, [("_committees", "rebuilt_invalid_committees_dict", len(state["committees"]))]

    for cid, c in list(committees.items()):
        if not isinstance(c, dict):
            committees[cid] = copy.deepcopy(DEFAULT_COMMITTEES.get(cid, {}))
            c = committees[cid]
            repairs.append((cid, "rebuilt_invalid_committee", ""))

        c.setdefault("name", str(cid).replace("_", " ").title())
        if not isinstance(c.get("name"), str) or not c["name"].strip():
            c["name"] = str(cid).replace("_", " ").title()
            repairs.append((cid, "committee_name_filled", c["name"]))

        try:
            c["layer"] = int(c.get("layer", 0))
        except Exception:
            c["layer"] = 0
            repairs.append((cid, "committee_layer_coerced", 0))

        members = c.get("members", [])
        if isinstance(members, str):
            members = [members]
        if not isinstance(members, list):
            members = []
        members = [aid for aid in members if aid in active]
        if not members:
            members = _find_arm_ids_by_keywords(state, COMMITTEE_NAME_KEYWORDS.get(cid, []))[:4]
        if not members and active:
            members = [next(iter(active))]
        # Stable de-dup.
        deduped = []
        for aid in members:
            if aid not in deduped:
                deduped.append(aid)
        c["members"] = deduped

        leader = c.get("leader")
        if leader not in active:
            leader = deduped[0] if deduped else None
            repairs.append((cid, "committee_leader_repaired", leader))
        if leader and leader not in deduped:
            deduped.insert(0, leader)
        c["leader"] = leader
        c["members"] = deduped

        c.setdefault("routing_hint", "")
        c.setdefault("output_policy", "Release compact findings to the next routing layer or Speakerbot.")
        c.setdefault("status", "active")
        if c.get("status") == "retired":
            c["retired"] = True
        c["status"] = "retired" if c.get("retired", False) else "active"

    cids = set(committees)
    for cid, c in committees.items():
        children = c.get("children", [])
        if isinstance(children, str):
            children = [children]
        if not isinstance(children, list):
            children = []
        c["children"] = [x for x in children if x in cids and x != cid]
        if c.get("parent") not in cids:
            c["parent"] = None

    # Derived membership index on arms. It is redundant but useful in prompts/debugging.
    for aid, arm in state.get("arms", {}).items():
        if not isinstance(arm, dict):
            continue
        arm["committees"] = sorted([cid for cid, c in committees.items()
                                   if aid in c.get("members", []) and c.get("status", "active") != "retired"])
        leader_of = sorted([cid for cid, c in committees.items()
                            if c.get("leader") == aid and c.get("status", "active") != "retired"])
        if leader_of:
            arm["committee_leader_of"] = leader_of
        elif "committee_leader_of" in arm:
            arm.pop("committee_leader_of", None)

    return state, repairs

def active_committee_ids(state):
    committees = state.get("committees", {}) if isinstance(state.get("committees"), dict) else {}
    return [cid for cid, c in committees.items()
            if isinstance(c, dict)
            and not c.get("retired", False)
            and c.get("status", "active") != "retired"
            and c.get("members")]

def committee_member_ids(state, committee_id, include_leader=True):
    c = (state.get("committees") or {}).get(committee_id, {})
    ids = list(c.get("members", []) or [])
    if include_leader and c.get("leader") and c["leader"] not in ids:
        ids.insert(0, c["leader"])
    active = set(_active_existing_arm_ids(state))
    return [aid for aid in ids if aid in active]

def committee_brief(state, committee_id):
    c = (state.get("committees") or {}).get(committee_id, {})
    leader = c.get("leader")
    return {
        "committee_id": committee_id,
        "name": c.get("name", committee_id),
        "layer": c.get("layer", 0),
        "leader": leader,
        "leader_name": state.get("arms", {}).get(leader, {}).get("name") if leader else None,
        "members": [
            {"arm_id": aid, "name": state["arms"][aid].get("name"), "lens": state["arms"][aid].get("lens")}
            for aid in committee_member_ids(state, committee_id)
        ],
        "parent": c.get("parent"),
        "children": c.get("children", []),
        "routing_hint": c.get("routing_hint", ""),
        "output_policy": c.get("output_policy", ""),
    }

def committee_report(state):
    state, _ = sanitize_committees(copy.deepcopy(state))
    lines = []
    for cid in active_committee_ids(state):
        c = state["committees"][cid]
        leader = c.get("leader")
        leader_name = state.get("arms", {}).get(leader, {}).get("name", leader)
        member_names = [f"{aid}:{state['arms'][aid].get('name')}" for aid in committee_member_ids(state, cid)]
        lines.append(
            f"- {cid} / {c.get('name')} (layer={c.get('layer')}, leader={leader}:{leader_name}, "
            f"parent={c.get('parent')}, children={c.get('children', [])}) members={member_names}; "
            f"routing_hint={c.get('routing_hint')}"
        )
    return "\n".join(lines) if lines else "(no active committees)"

def show_committees(state=None):
    state = state or load_architecture_state()
    print(committee_report(state))

def fresh_architecture_state():
    arch_id = short_id("arch")
    state = {
        "architecture_id": arch_id,
        "architecture_version": 1,
        "created_at": now_iso(),
        "updated_at": now_iso(),
        "globals": copy.deepcopy(DEFAULT_GLOBALS),
        "listener": {
            "model": LISTENER_MODEL,
            "temperature": LISTENER_TEMPERATURE,
            "role": "Reinterprets/thesaurizes the user prompt into separate near-identical inputs for each arm, and forwards liveness_evidence.",
        },
        "speaker": {
            "model": SPEAKER_MODEL,
            "role": "Reads arm reports and runs the PLAN->FORWARD->GATE->(FIX-ONLY CONTINUE|READY) FSM; renders the whole final answer atomically.",
            "atomic_final_render": True,
        },
        "evolution": {
            "model": EVOLUTION_MODEL,
            "temperature": EVOLUTION_TEMPERATURE,
            "allowed_mutations": [
                "adjust_global", "add_arm", "retire_arm", "rename_arm",
                "adjust_arm_parameter", "add_seed_memory",
                "adjust_listener_temperature", "adjust_speaker_model",
                "add_committee", "update_committee", "retire_committee",
                "set_committee_leader", "assign_arm_to_committee", "remove_arm_from_committee",
            ],
        },
        "arms": copy.deepcopy(DEFAULT_ARMS),
        "seed_memories": [
            {
                "content": "For exact-line-count requests: produce the entire answer in one atomic final render. Hard gate before emitting: count non-empty lines == requested N; no duplicate numbering; no repeated lines; no extra preface/epilogue. If gate fails, rewrite internally and re-check.",
                "importance": 0.95,
                "tags": ["formatting", "liveness", "exact-constraints", "speaker-gate"],
                "created_at": now_iso(),
            },
            {
                "content": "LIVENESS GATE (informational): A code function computes a single liveness result per step from {speaker_msg_id, speaker_timestamp, present, matched, missing_reason} and exposes it to bots as liveness_gate: PASS or FAIL(reason). This is the only gate; bots must not define additional 'required fields' or refuse to finalize over self-invented conditions. The code, not the model, enforces the block. If a bot believes a block is spurious it may set request_gate_override; a 66% arm quorum can lift a spurious block, but a real code-gate failure cannot be overridden by anyone.",
                "importance": 0.95,
                "tags": ["liveness", "gating", "determinism", "self-description"],
                "created_at": now_iso(),
            },
            {
                "content": "This architecture uses a listenerbot, multiple arms with personal memory, a shared scratch debate space, a speakerbot with an atomic final render and a code-enforced liveness FSM, and an evolutionbot that proposes bounded, type-checked mutations. Bots reason with awareness of this structure.",
                "importance": 0.90,
                "tags": ["self-description", "architecture-awareness"],
                "created_at": now_iso(),
            },
            {
                "content": "ARM DIVERSITY IS A HARD CONSTRAINT. The starfish needs genuinely different perspectives -- not a monoculture of audit/compliance specialists. If more than half the active arms share the same archetype (low-temperature, high-precision, audit-focused), Evolutionbot MUST retire the most redundant one before adding anything new. New arms should differ from every existing arm in at least TWO of: temperature, creativity, doubt, and lens domain. Healthy roster example: one literalist, one historian, one mechanic, one adversary, one dreamer, one domain specialist -- NOT six variants of schema auditor.",
                "importance": 0.95,
                "tags": ["diversity", "anti-monoculture", "evolution", "architecture-health"],
                "created_at": now_iso(),
            },
            {
                "content": "SEED MEMORY HYGIENE: Do not add seed memories that restate existing rules in slightly different words. Before proposing add_seed_memory, check whether the idea is already captured. If it is, do not add a variant. The system caps seed memories and will compact duplicates automatically.",
                "importance": 0.90,
                "tags": ["memory", "hygiene", "evolution", "dedup"],
                "created_at": now_iso(),
            },
        ],
    }
    state["committees"] = build_default_committees_for_state(state)
    return state

KNOWN_GOOD_PATH = STORAGE_ROOT / "architecture_last_known_good.json"
CORRUPTED_DIR = STORAGE_ROOT / "corrupted"
CORRUPTED_DIR.mkdir(parents=True, exist_ok=True)

# Globals that MUST be numeric, and the arm/tuning fields that must hold given types.
_NUMERIC_GLOBALS = [
    "MAX_OUTPUT_STEPS", "MAX_DELIBERATION_ROUNDS_PER_STEP", "MIN_DELIBERATION_STEPS",
    "MAX_TOTAL_API_CALLS", "FINAL_RENDER_WORD_LIMIT", "EARLY_FINAL_IF_MEAN_CONFIDENCE_GTE",
    "CONTINUE_PENALTY_AFTER_STEP", "MAX_CONTINUE_ACTIONS", "SHARED_MEMORY_TOP_K",
    "PERSONAL_MEMORY_TOP_K", "SHARED_SCRATCH_RECENT_K", "LOCAL_CONTEXT_RECENT_K",
    "MAX_ARM_COUNT", "MAX_NEW_ARMS_PER_EVOLUTION",
]
_LIST_GLOBALS = ["LIVENESS_GATE_REQUIRED_FIELDS"]

def validate_architecture(state):
    """Return (ok, errors). Pure structural type-checking -- the thing that would have
    caught the gen-5 brick. No API calls. Never raises."""
    errors = []
    if not isinstance(state, dict):
        return False, ["state is not a dict"]
    for top in ("globals", "arms", "listener", "speaker", "evolution"):
        if not isinstance(state.get(top), dict):
            errors.append(f"missing/!dict top-level: {top}")
    g = state.get("globals", {})
    if isinstance(g, dict):
        for k in _NUMERIC_GLOBALS:
            if k in g and not isinstance(g[k], (int, float, bool)):
                errors.append(f"globals.{k} must be numeric, got {type(g[k]).__name__}")
        for k in _LIST_GLOBALS:
            if k in g and not isinstance(g[k], list):
                errors.append(f"globals.{k} must be list, got {type(g[k]).__name__}")
    arms = state.get("arms", {})
    if isinstance(arms, dict):
        active = 0
        for aid, arm in arms.items():
            if not isinstance(arm, dict):
                errors.append(f"arm {aid} is not a dict"); continue
            if not isinstance(arm.get("name"), str):
                errors.append(f"arm {aid}.name must be str, got {type(arm.get('name')).__name__}")
            if not isinstance(arm.get("lens"), str):
                errors.append(f"arm {aid}.lens must be str")
            if not isinstance(arm.get("tuning"), dict):
                errors.append(f"arm {aid}.tuning must be dict")
            else:
                for tk, tv in arm["tuning"].items():
                    if not isinstance(tv, (int, float)):
                        errors.append(f"arm {aid}.tuning.{tk} must be numeric")
            if not isinstance(arm.get("parents", []), list):
                errors.append(f"arm {aid}.parents must be list")
            if not arm.get("retired", False) and arm.get("status", "active") != "retired":
                active += 1
        if active < 2:
            errors.append(f"need >=2 active arms, found {active}")
    committees = state.get("committees", {})
    if committees is not None and not isinstance(committees, dict):
        errors.append("committees must be dict when present")
    if isinstance(committees, dict):
        active_ids = set(_active_existing_arm_ids(state))
        for cid, c in committees.items():
            if not isinstance(c, dict):
                errors.append(f"committee {cid} is not a dict"); continue
            if c.get("status", "active") == "retired" or c.get("retired", False):
                continue
            if c.get("leader") and c.get("leader") not in active_ids:
                errors.append(f"committee {cid}.leader points to inactive/missing arm {c.get('leader')}")
            members = c.get("members", [])
            if not isinstance(members, list):
                errors.append(f"committee {cid}.members must be list")
            else:
                bad = [aid for aid in members if aid not in active_ids]
                if bad:
                    errors.append(f"committee {cid}.members contains inactive/missing arms {bad[:3]}")
    return (len(errors) == 0), errors

def smoke_test_state(state):
    """Dry-run the hot paths against a candidate state so a mutation that would crash at
    runtime is rejected at write time. Structural only, no API calls. Returns (ok, error)."""
    try:
        ok, errs = validate_architecture(state)
        if not ok:
            return False, "; ".join(errs[:5])
        ids = active_arm_ids(state)
        if not ids:
            return False, "no active arms"
        _ = architecture_report(state)  # exercises every arm/committee field the way prompts do
        _ = committee_report(state)
        for cid in active_committee_ids(state):
            _ = committee_brief(state, cid)
        for aid in ids:
            arm = state["arms"][aid]
            _ = f"{arm['name']}::{arm['lens']}::{arm['tuning'].get('temperature')}"
            _ = arm.get("web_access", True)
        # Exercise a representative global read path.
        _ = float(state["globals"].get("EARLY_FINAL_IF_MEAN_CONFIDENCE_GTE", 0))
        _ = int(state["globals"].get("MAX_OUTPUT_STEPS", 1))
        return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def mark_known_good(state):
    """Stamp a state as the last architecture that demonstrably ran end to end."""
    ok, _ = smoke_test_state(state)
    if ok:
        atomic_write_json(KNOWN_GOOD_PATH, {"saved_at": now_iso(), "state": state})

def quarantine_state(raw, reason):
    """Move a poisoned state out of the load path (never delete it)."""
    qpath = CORRUPTED_DIR / f"corrupt_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}.json"
    atomic_write_json(qpath, {"quarantined_at": now_iso(), "reason": reason, "raw_state": raw})
    return qpath

def _load_known_good():
    kg = load_json(KNOWN_GOOD_PATH)
    if kg and isinstance(kg.get("state"), dict):
        return kg["state"]
    return None

def save_architecture_state(state, reason="manual_save"):
    state = copy.deepcopy(state)
    state["updated_at"] = now_iso()
    atomic_write_json(ARCH_STATE_PATH, state)
    snap = ARCH_DIR / f"{state['architecture_id']}_v{state['architecture_version']}_{uuid.uuid4().hex[:6]}.json"
    atomic_write_json(snap, {"reason": reason, "saved_at": now_iso(), "state": state})
    return state, snap

def load_architecture_state():
    raw = load_json(ARCH_STATE_PATH)
    if not raw:
        state = fresh_architecture_state()
        save_architecture_state(state, reason="initial_create")
        mark_known_good(state)
        return state

    # 1) Sanitize known shapes first.
    try:
        state, repairs = sanitize_arms(copy.deepcopy(raw))
        state, committee_repairs = sanitize_committees(state)
        repairs = list(repairs) + list(committee_repairs)
    except Exception as e:
        state, repairs = raw, [("_", f"sanitize_failed:{e}", "")]

    # 2) Validate + smoke-test. If it would crash at runtime, auto-revert (your
    #    'can't even restart' fix): quarantine the poison and load last-known-good.
    ok, err = smoke_test_state(state)
    if not ok:
        qpath = quarantine_state(raw, f"load-time validation failed: {err}")
        rprint(f"[red]Corrupt architecture_state.json detected: {err}[/red]")
        rprint(f"[red]Quarantined to {qpath.name}; reverting to last known-good.[/red]")
        kg = _load_known_good()
        if kg is not None:
            kg2, _ = sanitize_arms(copy.deepcopy(kg))
            kg2, _ = sanitize_committees(kg2)
            ok2, err2 = smoke_test_state(kg2)
            if ok2:
                kg2, _ = save_architecture_state(kg2, reason="auto_revert_to_known_good")
                rprint(f"[green]Reverted to known-good {kg2['architecture_id']} v{kg2['architecture_version']}.[/green]")
                return kg2
            rprint(f"[red]Known-good also failed validation ({err2}); rebuilding fresh.[/red]")
        else:
            rprint("[yellow]No known-good checkpoint exists; rebuilding fresh architecture.[/yellow]")
        state = fresh_architecture_state()
        save_architecture_state(state, reason="auto_revert_fresh_rebuild")
        mark_known_good(state)
        return state

    if repairs:
        state, _ = save_architecture_state(state, reason="sanitize_on_load")
        rprint(f"[yellow]Sanitized {len(repairs)} arm metadata issue(s) on load.[/yellow]")
    return state

def list_architecture_snapshots(limit=20):
    snaps = sorted(ARCH_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    out = []
    for p in snaps[:limit]:
        d = load_json(p, {})
        st = d.get("state", {})
        out.append({"file": p.name, "version": st.get("architecture_version"),
                    "arch_id": st.get("architecture_id"), "reason": d.get("reason"),
                    "saved_at": d.get("saved_at")})
    return out

def revert_architecture(to_version=None, confirm=False):
    """Manual failsafe undo. Quarantines the current state, then restores either the last
    known-good (default) or a specific snapshot version. Nothing is ever deleted."""
    current = load_json(ARCH_STATE_PATH)
    if not confirm:
        kg = _load_known_good()
        print("revert_architecture() will quarantine the CURRENT state and restore:")
        if to_version is None:
            kgs = (kg or {}).get("architecture_version") if kg else None
            print(f"  -> last known-good (version {kgs})" if kg else "  -> (no known-good found; will rebuild fresh)")
        else:
            print(f"  -> snapshot version {to_version}")
        print("Call again with confirm=True to proceed.")
        return None

    if current:
        quarantine_state(current, f"manual revert (to_version={to_version})")

    target = None
    if to_version is None:
        target = _load_known_good()
    else:
        for p in sorted(ARCH_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True):
            d = load_json(p, {})
            if (d.get("state") or {}).get("architecture_version") == to_version:
                target = d["state"]; break

    if target is None:
        print("No matching target found; rebuilding fresh.")
        state = fresh_architecture_state()
        save_architecture_state(state, reason="manual_revert_fresh")
        mark_known_good(state)
    else:
        state, _ = sanitize_arms(copy.deepcopy(target))
        state, _ = sanitize_committees(state)
        ok, err = smoke_test_state(state)
        if not ok:
            print(f"Target failed validation ({err}); rebuilding fresh.")
            state = fresh_architecture_state()
        state, _ = save_architecture_state(state, reason=f"manual_revert(to_version={to_version})")
    print(f"Reverted. Current: {state['architecture_id']} v{state['architecture_version']}")
    show_architecture(state)
    return state

def active_arm_ids(state):
    return [k for k, v in state["arms"].items()
            if not v.get("retired", False) and v.get("status", "active") != "retired"]

def architecture_report(state):
    """Injected into prompts so every bot knows how the system currently works."""
    arms_lines = []
    for arm_id in active_arm_ids(state):
        arm = state["arms"][arm_id]
        t = arm.get("tuning", {})
        arms_lines.append(
            f"- {arm_id} / {arm.get('name')}: lens={arm.get('lens')}; "
            f"web_access={arm.get('web_access', True)}; "
            f"generation={arm.get('generation', 0)}; parents={arm.get('parents', [])}; "
            f"temperature={t.get('temperature')}; doubt={t.get('doubt')}; "
            f"precision={t.get('precision')}; creativity={t.get('creativity')}"
        )
    g = state["globals"]
    return f"""
CURRENT PENTAMIND ARCHITECTURE REPORT
architecture_id: {state.get('architecture_id')}
version: {state.get('architecture_version')}
active_arms: {len(active_arm_ids(state))}

How the system currently works:
1. Listenerbot rewrites the user prompt into a global summary, proposed committee meeting order, committee-level charges, and fallback arm-specific heard prompts.
2. Committee leaders receive the committee charge and translate it into increasingly specific member prompts.
3. Arms can belong to multiple committees and multiple layers. Each arm receives: the user prompt, its member prompt, committee context, this architecture report, recent shared scratch, shared + personal memory, and liveness_evidence.
4. Committees may be nested; a leader may route findings to child/sibling/parent committees before information is released to Speakerbot.
5. Each arm emits a compact structured report: interpretation, stance, confidence, candidate answer/patch, risks/falsifiers, challenges, liveness_ack, and ready_to_final_render.
6. Speakerbot runs a code-enforced FSM: PLAN -> FORWARD -> GATE -> (FIX-ONLY CONTINUE_DELIBERATION | READY). FINAL_RENDER is blocked while the liveness gate fails.
7. FINAL_RENDER is atomic: the final answer replaces the draft as one complete artifact. No appended fragments.
8. Evolutionbot runs after diagnostics and proposes bounded, type-checked mutations (importance labels coerced; add_arm specs normalized).
9. Every run gets a unique run directory; nothing overwrites prior data.

Current liveness/rendering globals:
- MAX_OUTPUT_STEPS={g.get('MAX_OUTPUT_STEPS')}
- MAX_DELIBERATION_ROUNDS_PER_STEP={g.get('MAX_DELIBERATION_ROUNDS_PER_STEP')}
- LIVENESS_GATE_ENABLED={g.get('LIVENESS_GATE_ENABLED')}
- ATOMIC_FINAL_RENDER={g.get('ATOMIC_FINAL_RENDER')}
- EXACT_FORMAT_HARD_GATE={g.get('EXACT_FORMAT_HARD_GATE')}
- FINAL_RENDER_WORD_LIMIT={g.get('FINAL_RENDER_WORD_LIMIT')}
- EARLY_FINAL_IF_MEAN_CONFIDENCE_GTE={g.get('EARLY_FINAL_IF_MEAN_CONFIDENCE_GTE')}

Active committees:
{committee_report(state)}

Active arms:
{chr(10).join(arms_lines)}
""".strip()

def show_architecture(state=None):
    state = state or load_architecture_state()
    print(architecture_report(state))

state = load_architecture_state()
state, snap = save_architecture_state(state, reason="notebook_loaded")
rprint(f"[green]Architecture loaded:[/green] {state['architecture_id']} v{state['architecture_version']}")
show_architecture(state)


In [ ]:
# =========================
# Memory helpers
# =========================

def personal_memory_path(arm_id):
    return MEMORY_DIR / f"{arm_id}_personal_memory.jsonl"

def add_shared_memory(content, importance=0.5, tags=None, source="manual"):
    row = {
        "memory_id": short_id("smem"),
        "created_at": now_iso(),
        "source": source,
        "content": content,
        "importance": coerce_importance(importance),
        "tags": tags or [],
    }
    append_jsonl(SHARED_MEMORY_PATH, row)
    return row

def add_personal_memory(arm_id, content, importance=0.5, tags=None, source="manual"):
    row = {
        "memory_id": short_id(f"{arm_id}_pmem"),
        "created_at": now_iso(),
        "source": source,
        "arm_id": arm_id,
        "content": content,
        "importance": coerce_importance(importance),
        "tags": tags or [],
    }
    append_jsonl(personal_memory_path(arm_id), row)
    return row

def retrieve_memory(path, query, top_k=3):
    """Tiny local retrieval: keyword overlap + importance. Fine for a prototype."""
    rows = read_jsonl(path)
    if not rows:
        return []
    q_words = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    scored = []
    for row in rows:
        text = (row.get("content", "") + " " + " ".join(row.get("tags", []))).lower()
        words = set(re.findall(r"[a-zA-Z0-9_]+", text))
        overlap = len(q_words & words)
        score = overlap + coerce_importance(row.get("importance", 0)) * 2.0
        scored.append((score, row))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for s, r in scored[:top_k]]

def ensure_seed_memories(state):
    existing = read_jsonl(SHARED_MEMORY_PATH)
    existing_hashes = {sha256_text(r.get("content", "")) for r in existing}
    for mem in state.get("seed_memories", []):
        h = sha256_text(mem["content"])
        if h not in existing_hashes:
            add_shared_memory(mem["content"], mem.get("importance", 0.5),
                              mem.get("tags", []), source="architecture_seed")

def compact_shared_memory(max_entries=200):
    """Deduplicate the shared memory JSONL file by content hash.
    Keeps the first occurrence of each unique content. Hard-caps at max_entries."""
    rows = read_jsonl(SHARED_MEMORY_PATH)
    if not rows:
        return 0
    seen_hashes = set()
    deduped = []
    for row in rows:
        h = sha256_text(row.get("content", ""))
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        deduped.append(row)
    if len(deduped) > max_entries:
        deduped = deduped[-max_entries:]
    removed = len(rows) - len(deduped)
    if removed > 0:
        tmp = SHARED_MEMORY_PATH.with_suffix(".jsonl.compacting")
        with tmp.open("w", encoding="utf-8") as f:
            for row in deduped:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        tmp.replace(SHARED_MEMORY_PATH)
    return removed

def compact_seed_memories(state, max_seeds=None):
    """Deduplicate seed_memories in the architecture state by content hash.
    Keeps the first occurrence. Caps at max_seeds (highest importance wins)."""
    max_seeds = max_seeds or state.get("globals", {}).get("MAX_SEED_MEMORIES", 8)
    seeds = state.get("seed_memories", [])
    seen = set()
    unique = []
    for s in seeds:
        h = sha256_text(s.get("content", ""))
        if h not in seen:
            seen.add(h)
            unique.append(s)
    if len(unique) > max_seeds:
        scored = sorted(enumerate(unique), key=lambda x: -coerce_importance(x[1].get("importance", 0.5)))
        keep_indices = set(idx for idx, _ in scored[:max_seeds])
        unique = [s for i, s in enumerate(unique) if i in keep_indices]
    state["seed_memories"] = unique
    return state

ensure_seed_memories(load_architecture_state())
compact_shared_memory()  # Clean up any dupes from prior runs.
rprint("[green]Memory layer ready (with compaction).[/green]")


In [ ]:
# =========================
# OpenAI call wrappers (v0.5: correct tool name + real fallback)
# =========================

API_CALLS_USED = 0

def reset_api_counter():
    global API_CALLS_USED
    API_CALLS_USED = 0

def _build_kwargs(model, system, user, temperature, max_output_tokens, use_tools):
    kwargs = {
        "model": model,
        "input": [
            {"role": "system", "content": sanitize_text_for_api(system)},
            {"role": "user", "content": sanitize_text_for_api(user)},
        ],
    }
    if use_tools and ENABLE_WEB_SEARCH:
        kwargs["tools"] = [WEB_SEARCH_TOOL]
    if temperature is not None:
        kwargs["temperature"] = temperature
    if max_output_tokens:
        kwargs["max_output_tokens"] = max_output_tokens
    return kwargs

def call_openai_text(model, system, user, temperature=0.3, max_output_tokens=None,
                     fallback_model=FALLBACK_MODEL, use_tools=True):
    """Responses API text wrapper.

    v0.4 bugs fixed:
      - tool was named 'websearch' (invalid) -> now 'web_search'.
      - the fallback path re-sent the same failing kwargs (incl. temperature/tools)
        so it died the same way. Now each retry strips one likely culprit:
        attempt 1: requested model + tools + temperature
        attempt 2: requested model, no tools (tool/model mismatch)
        attempt 3: requested model, no tools, no temperature
        attempt 4: fallback model, no tools, no temperature
    """
    global API_CALLS_USED
    API_CALLS_USED += 1
    attempts = [
        dict(model=model, use_tools=True,  temperature=temperature),
        dict(model=model, use_tools=False, temperature=temperature),
        dict(model=model, use_tools=False, temperature=None),
        dict(model=fallback_model, use_tools=False, temperature=None),
    ]
    last_err = None
    for i, a in enumerate(attempts):
        try:
            kwargs = _build_kwargs(a["model"], system, user, a["temperature"],
                                   max_output_tokens, a["use_tools"])
            resp = client.responses.create(**kwargs)
            raw_text = getattr(resp, "output_text", str(resp))
            # Force output to ASCII immediately so non-ASCII never enters the transcript.
            return sanitize_text_for_api(raw_text), a["model"]
        except Exception as e:
            last_err = e
            rprint(f"[red]API attempt {i+1}/4 failed ({a['model']}, tools={a['use_tools']}): "
                   f"{type(e).__name__}: {str(e)[:200]}[/red]")
            continue
    rprint(f"[bold red]ALL 4 API ATTEMPTS FAILED. Last error: {type(last_err).__name__}: {last_err}[/bold red]")
    return json.dumps({"_error": str(last_err)}), model

def call_openai_json(model, system, user, temperature=0.3, max_output_tokens=None,
                     fallback_model=FALLBACK_MODEL, use_tools=True):
    sys2 = system + "\n\nReturn valid JSON only. No markdown fences. No prose outside JSON."
    text, used_model = call_openai_text(model, sys2, user, temperature=temperature,
                                        max_output_tokens=max_output_tokens,
                                        fallback_model=fallback_model, use_tools=use_tools)
    obj = extract_json_object(text)
    # Deep-scrub the parsed object so no non-ASCII string survives into the transcript.
    obj = force_ascii_deep(obj)
    if is_error_obj(obj):
        err_detail = obj.get("_error") or obj.get("_raw") or obj.get("_parse_error")
        rprint(f"[red]JSON parse/call error: {tokenish_truncate(str(err_detail), 300)}[/red]")
    return obj, used_model, text

rprint("[green]OpenAI wrappers ready (web_search tool, graduated fallback).[/green]")



In [ ]:
# =========================
# Exact-format detection and validation
# =========================

def infer_exact_line_request(prompt):
    p = (prompt or "").lower()
    patterns = [
        r"exactly\s+(\d+)\s+(?:non-empty\s+)?lines?",
        r"(\d+)\s*[- ]line\b",
        r"give\s+(?:me\s+)?(?:a\s+)?(\d+)\s+line\b",
        r"output\s+(?:exactly\s+)?(\d+)\s+(?:non-empty\s+)?lines?",
    ]
    for pat in patterns:
        m = re.search(pat, p)
        if m:
            return int(m.group(1))
    return None

def nonempty_lines(text):
    return [ln.strip() for ln in str(text).splitlines() if ln.strip()]

def validate_exact_lines(text, n):
    lines = nonempty_lines(text)
    seen_nums = []
    for ln in lines:
        m = re.match(r"^(\d+)[\)\.]", ln)
        if m:
            seen_nums.append(int(m.group(1)))
    duplicate_numbering = len(seen_nums) != len(set(seen_nums))
    repeated_lines = len(lines) != len(set(lines))
    return {
        "requested_lines": n,
        "actual_nonempty_lines": len(lines),
        "ok_line_count": len(lines) == n,
        "duplicate_numbering": duplicate_numbering,
        "repeated_lines": repeated_lines,
        "ok": len(lines) == n and not duplicate_numbering and not repeated_lines,
        "lines": lines,
    }

def exact_format_instruction(prompt):
    n = infer_exact_line_request(prompt)
    if not n:
        return ""
    return f"""
EXACT FORMAT HARD GATE:
The user requested exactly {n} non-empty lines.
When FINAL_RENDERing:
- Output exactly {n} non-empty lines.
- If numbered, use each number once.
- No preface, no epilogue, no duplicated lines, no partial continuation.
- Internally validate line_count == {n} before returning FINAL_RENDER.
- If your draft fails validation, rewrite internally before returning.
""".strip()

rprint("[green]Format validators ready.[/green]")


In [ ]:
# =========================
# Liveness gate (v0.5 core; notes items 1-3)
# =========================
#
# matched is computed HERE, in code -- never trusted from the model. The Listenerbot
# forwards the immediately preceding Speakerbot emission's id/timestamp into each arm
# payload; this module decides present/matched deterministically.

def speaker_emission_signature(speaker_decision):
    """Stable id/timestamp for a Speakerbot emission, used as the liveness anchor."""
    if not speaker_decision:
        return None, None
    msg_id = speaker_decision.get("speaker_msg_id")
    ts = speaker_decision.get("speaker_timestamp")
    return msg_id, ts

def build_liveness_evidence(prev_speaker_decision, forwarded):
    """Compute liveness_evidence deterministically.

    prev_speaker_decision: the immediately preceding Speakerbot emission (or None on step 1).
    forwarded: the {speaker_msg_id, speaker_timestamp} that the Listenerbot actually placed
               into the arm payload.
    """
    exp_id, exp_ts = speaker_emission_signature(prev_speaker_decision)
    fwd_id = (forwarded or {}).get("speaker_msg_id")
    fwd_ts = (forwarded or {}).get("speaker_timestamp")

    # Step 1 (no prior speaker emission yet) is a legitimate "present, matched" baseline.
    if prev_speaker_decision is None:
        return {
            "speaker_msg_id": fwd_id or "BOOTSTRAP",
            "speaker_timestamp": fwd_ts or now_iso(),
            "present": True,
            "matched": True,
            "missing_reason": "",
        }

    present = bool(fwd_id) or bool(fwd_ts)
    if exp_id and fwd_id:
        matched = (fwd_id == exp_id)
        reason = "" if matched else "speaker_msg_id mismatch"
    elif exp_ts and fwd_ts:
        matched = (fwd_ts == exp_ts)  # exact string equality, no rounding
        reason = "" if matched else "speaker_timestamp mismatch"
    else:
        matched = False
        reason = "no speaker_msg_id or speaker_timestamp available to match"

    if not present:
        reason = "no forwarded speaker id/timestamp present"

    return {
        "speaker_msg_id": fwd_id or "",
        "speaker_timestamp": fwd_ts or "",
        "present": present,
        "matched": bool(matched),
        "missing_reason": reason,
    }

def liveness_ok(evidence, required_fields):
    if not isinstance(evidence, dict):
        return False, "liveness_evidence missing"
    for f in required_fields:
        if f not in evidence or evidence.get(f) is None:
            return False, f"missing field: {f}"
    if not evidence.get("present"):
        return False, "present=false"
    if not evidence.get("matched"):
        return False, evidence.get("missing_reason") or "matched=false"
    return True, ""

def evaluate_override(arm_reports, code_gate_passes, required_fields, quorum=0.66):
    """Decide whether a 66% arm quorum may lift a SPURIOUS block.

    Returns dict with: allowed (bool), votes, voters, ratio, block_kind, and detail.

    Hard rule: an override is ONLY ever allowed when the code gate already passes.
    A real code-gate failure (matched=false / missing required field) is never
    overridable by any number of votes -- the predicate short-circuits below.

    block_kind classifies WHY arms were stuck (logged either way, per user choice):
      - "fake_field": at least one arm asserted a 'required'/missing field whose name
        is NOT in required_fields (e.g. web_research_packaged) -> a hallucinated gate.
      - "non_convergence": no fake field cited; arms simply kept ready=false.
      - "none": nothing to override.
    """
    active = [r for r in arm_reports]
    n = len(active)
    voters = []
    for r in active:
        req = r.get("request_gate_override") or {}
        val = str(req.get("value", "")).strip().lower() == "true"
        if val:
            voters.append(r.get("arm_id"))
    votes = len(voters)
    ratio = (votes / n) if n else 0.0

    # Classify the block kind by scanning arm text for invented required-field names.
    fake_field = None
    field_pat = re.compile(r"([a-z][a-z0-9_]{3,})", re.I)
    known = set(required_fields)
    blob_keys = ("missing_reason", "scratch", "interpretation", "candidate_patch_or_phrase")
    for r in active:
        if str(r.get("ready_to_final_render", "")).lower() == "true":
            continue
        text_bits = [str(r.get(k, "")) for k in blob_keys]
        live = r.get("liveness_evidence") or {}
        text_bits.append(str(live.get("missing_reason", "")))
        text = " ".join(text_bits).lower()
        # Heuristic: a token paired with 'field'/'missing'/'required'/'packaged' that isn't a real field.
        for m in field_pat.findall(text):
            if m in known:
                continue
            if m in ("field", "fields", "required", "missing", "gate", "liveness", "present", "matched", "reason"):
                continue
            if any(kw in text for kw in ("required", "missing", "packaged", "carrier", "propagation", "must include")):
                fake_field = m
                break
        if fake_field:
            break

    if not code_gate_passes:
        block_kind = "real_gate"  # not overridable
    elif fake_field:
        block_kind = "fake_field"
    elif votes > 0 or any(str(r.get("ready_to_final_render", "")).lower() != "true" for r in active):
        block_kind = "non_convergence"
    else:
        block_kind = "none"

    allowed = bool(code_gate_passes) and (ratio >= quorum)
    return {
        "allowed": allowed, "votes": votes, "voters": voters, "active_arms": n,
        "ratio": round(ratio, 3), "quorum": quorum, "block_kind": block_kind,
        "fake_field": fake_field,
        "detail": ("override blocked: real code-gate failure" if not code_gate_passes
                   else f"{votes}/{n} arms voted override ({block_kind})"),
    }

rprint("[green]Liveness gate ready.[/green]")


In [ ]:
# =========================
# Bots + committee routing substrate
# =========================

def normalize_meeting_order(state, listener_output=None):
    """Return a safe, de-duped committee order.
    Listenerbot may suggest an order, but code normalizes it so bogus committee ids
    cannot crash the run."""
    committee_ids = active_committee_ids(state)
    if not committee_ids:
        return []
    by_layer = sorted(committee_ids, key=lambda cid: ((state.get("committees", {}).get(cid, {}) or {}).get("layer", 0), cid))
    proposed = []
    if isinstance(listener_output, dict):
        proposed = listener_output.get("meeting_order") or listener_output.get("committee_meeting_order") or []
    if isinstance(proposed, str):
        proposed = [proposed]
    order = []
    for cid in list(proposed) + by_layer:
        if cid in committee_ids and cid not in order:
            order.append(cid)
    return order

def committee_routing_neighbors(state, committee_id):
    """Default safe graph edges: children, then parent, then all higher-layer committees."""
    committees = state.get("committees", {}) or {}
    c = committees.get(committee_id, {}) or {}
    out = []
    for cid in c.get("children", []) or []:
        if cid in committees and cid not in out:
            out.append(cid)
    parent = c.get("parent")
    if parent in committees and parent not in out:
        out.append(parent)
    layer = int(c.get("layer", 0) or 0)
    for cid, other in sorted(committees.items(), key=lambda kv: (int((kv[1] or {}).get("layer", 0) or 0), kv[0])):
        if cid != committee_id and int((other or {}).get("layer", 0) or 0) > layer and cid not in out:
            out.append(cid)
    return [cid for cid in out if cid in active_committee_ids(state)]

def listenerbot(user_prompt, state, run_dir):
    arm_ids = active_arm_ids(state)
    committee_ids = active_committee_ids(state)
    arms_brief = {aid: {"name": state["arms"][aid]["name"], "lens": state["arms"][aid]["lens"],
                        "committees": state["arms"][aid].get("committees", [])}
                  for aid in arm_ids}
    committees_brief = {cid: committee_brief(state, cid) for cid in committee_ids}

    system = f"""
You are Listenerbot for Pentamind/Solasterid.

Your job:
- Preserve the user's task intent.
- Create a global summary.
- Propose a committee meeting order when committee routing is available.
- Translate the user prompt into committee-level charges. These should be broader than individual arm prompts.
- Provide fallback arm-specific heard prompts in case committee routing is unavailable.
- You are not the final answerer; you are the intake/router.

Committee routing model:
user -> Listenerbot -> committee leader -> committee members -> optional routed committees -> Speakerbot.
Each hop should become more specific, not more verbose.

{architecture_report(state)}
"""
    user = json.dumps({
        "user_prompt": user_prompt,
        "active_committees": committees_brief,
        "active_arms": arms_brief,
        "listener_temperature": state["listener"].get("temperature", LISTENER_TEMPERATURE),
        "required_json_shape": {
            "summary": "one sentence",
            "meeting_order": ["committee_id in desired order"],
            "per_committee": {
                "committee_id": {
                    "heard": "committee-level charge",
                    "why_this_committee": "short",
                    "suggested_routes": ["optional committee ids to consult later"],
                    "release_condition": "what this committee should produce before passing info onward",
                }
            },
            "per_arm": {"arm_id": {"heard": "fallback lens-specific prompt", "note": "what to emphasize"}},
        },
    }, ensure_ascii=False, indent=2)
    obj, used_model, raw = call_openai_json(
        state["listener"].get("model", LISTENER_MODEL),
        system, user,
        temperature=state["listener"].get("temperature", LISTENER_TEMPERATURE),
        max_output_tokens=2200,
        use_tools=LISTENER_WEB_ACCESS,
    )
    if is_error_obj(obj):
        err_detail = obj.get("_error") or obj.get("_raw") or str(obj.get("_parse_error"))
        rprint(f"[red]Listenerbot call error: {tokenish_truncate(str(err_detail), 200)}[/red]")
        obj = {"summary": user_prompt, "meeting_order": committee_ids, "per_committee": {}, "per_arm": {},
               "_call_error": err_detail}
    if "per_arm" not in obj or not isinstance(obj.get("per_arm"), dict):
        obj["per_arm"] = {}
    if "per_committee" not in obj or not isinstance(obj.get("per_committee"), dict):
        obj["per_committee"] = {}
    for aid in arm_ids:
        obj["per_arm"].setdefault(aid, {"heard": user_prompt, "note": "Fallback: preserve original user prompt."})
    for cid in committee_ids:
        c = state["committees"][cid]
        obj["per_committee"].setdefault(cid, {
            "heard": f"Handle the user prompt through the {c.get('name', cid)} committee mandate: {c.get('routing_hint', '')}",
            "why_this_committee": "Default committee charge.",
            "suggested_routes": committee_routing_neighbors(state, cid)[:2],
            "release_condition": c.get("output_policy", "Release compact findings."),
        })
    obj["meeting_order"] = normalize_meeting_order(state, obj)
    obj["_model_used"] = used_model
    save_run_file(run_dir, "listener_output.json", obj)
    return obj

def committee_leaderbot(committee_id, user_prompt, listener_output, state, shared_scratch,
                        step_index, deliberation_round, run_dir, transcript=None):
    """A committee leader converts the listener's committee charge into member-specific prompts
    and optional routing instructions. This is the user->listener->leader->member specificity ramp."""
    committees = state.get("committees", {}) or {}
    committee = committees.get(committee_id, {})
    leader_id = committee.get("leader")
    leader = state.get("arms", {}).get(leader_id, {})
    member_ids = committee_member_ids(state, committee_id)
    c_listener = (listener_output.get("per_committee") or {}).get(committee_id, {})
    committee_charge = c_listener.get("heard") or user_prompt
    recent_scratch = shared_scratch[-state["globals"].get("SHARED_SCRATCH_RECENT_K", 30):]

    system = f"""
You are the committee leader for {committee_id} / {committee.get('name', committee_id)}.
Leader arm: {leader_id} / {leader.get('name')}

Your job:
- Read the Listenerbot's committee-level charge.
- Decide what this committee should do now.
- Produce increasingly specific prompts for each committee member.
- Decide whether findings should be released to Speakerbot after member reports.
- Optionally route/bounce the problem to other committees after this one.
- Keep routing lean. Do not route everywhere by default.

You are not Speakerbot and not the final answerer.
You can route to parent/child/sibling/later committees, but only if they add real value.

Committee brief:
{json.dumps(committee_brief(state, committee_id), ensure_ascii=False, indent=2)}

Known safe routing neighbors:
{json.dumps(committee_routing_neighbors(state, committee_id), ensure_ascii=False)}

{architecture_report(state)}
"""
    payload = {
        "user_prompt": user_prompt,
        "committee_id": committee_id,
        "committee_charge": committee_charge,
        "listener_committee_packet": c_listener,
        "committee_members": [
            {"arm_id": aid, "name": state["arms"][aid].get("name"), "lens": state["arms"][aid].get("lens"),
             "other_committees": state["arms"][aid].get("committees", [])}
            for aid in member_ids
        ],
        "recent_shared_scratch": recent_scratch,
        "recent_speaker_decisions": (transcript or {}).get("speaker_decisions", [])[-3:],
        "step_index": step_index,
        "deliberation_round": deliberation_round,
        "required_json_shape": {
            "committee_action": "deliberate|skip|route_only",
            "leader_summary": "short",
            "meeting_order_note": "why this committee should act now",
            "per_member": {
                "arm_id": {
                    "heard": "member-specific prompt, narrower than committee charge",
                    "role_in_committee": "what this arm owns in this committee pass",
                    "expected_output": "specific artifact or finding requested",
                }
            },
            "route_to": ["optional committee ids to consult after this one"],
            "release_to_speaker": "true|false",
            "release_summary": "what information should leave the committee if released",
            "risks": ["short"],
        },
    }
    obj, used_model, raw = call_openai_json(
        ARM_MODEL, system,
        json.dumps(payload, ensure_ascii=False, indent=2),
        temperature=leader.get("tuning", {}).get("temperature", 0.35),
        max_output_tokens=1800,
        use_tools=bool(leader.get("web_access", True)),
    )
    if is_error_obj(obj):
        err_detail = obj.get("_error") or obj.get("_raw") or str(obj.get("_parse_error"))
        rprint(f"[red]Leaderbot call error ({committee_id}): {tokenish_truncate(str(err_detail), 200)}[/red]")
        obj = {"committee_action": "deliberate", "leader_summary": "leader call failed; using fallback member prompts",
               "per_member": {}, "route_to": [], "release_to_speaker": "true",
               "_call_error": err_detail}
    if not isinstance(obj.get("per_member"), dict):
        obj["per_member"] = {}
    for aid in member_ids:
        obj["per_member"].setdefault(aid, {
            "heard": committee_charge,
            "role_in_committee": "Fallback: answer through your normal arm lens within this committee.",
            "expected_output": "Compact committee finding.",
        })
    # Normalize routes so hallucinated committee ids cannot enter the queue.
    routes = obj.get("route_to") or []
    if isinstance(routes, str):
        routes = [routes]
    active_cids = set(active_committee_ids(state))
    obj["route_to"] = [cid for cid in routes if cid in active_cids and cid != committee_id]
    obj["_model_used"] = used_model
    obj["committee_id"] = committee_id
    obj["committee_name"] = committee.get("name", committee_id)
    obj["committee_leader_id"] = leader_id
    obj["committee_leader_name"] = leader.get("name")
    save_run_file(run_dir, f"committee_{step_index}_{deliberation_round}_{committee_id}.json", obj)
    return obj

def build_arm_prompt(arm_id, user_prompt, heard_prompt, state, shared_scratch,
                     liveness_evidence, transcript=None, committee_context=None):
    g = state["globals"]
    arm = state["arms"][arm_id]
    query = user_prompt + "\n" + heard_prompt
    if committee_context:
        query += "\n" + str(committee_context.get("committee_id", "")) + "\n" + str(committee_context.get("committee_charge", ""))
    shared_mem = retrieve_memory(SHARED_MEMORY_PATH, query, g["SHARED_MEMORY_TOP_K"])
    personal_mem = retrieve_memory(personal_memory_path(arm_id), query, g["PERSONAL_MEMORY_TOP_K"])
    recent_scratch = shared_scratch[-g["SHARED_SCRATCH_RECENT_K"]:]
    return {
        "architecture_report": architecture_report(state),
        "arm_id": arm_id,
        "arm_name": arm["name"],
        "arm_lens": arm["lens"],
        "arm_tuning": arm.get("tuning", {}),
        "arm_committees": arm.get("committees", []),
        "committee_context": committee_context or {},
        "web_access": arm.get("web_access", True),
        "user_prompt": user_prompt,
        "heard_prompt": heard_prompt,
        "shared_memory": shared_mem,
        "personal_memory": personal_mem,
        "recent_shared_scratch": recent_scratch,
        "recent_speaker_decisions": (transcript or {}).get("speaker_decisions", [])[-3:],
        # v0.5: liveness_evidence is a required top-level field in every arm payload.
        "liveness_evidence": liveness_evidence,
        "task": "Produce a compact arm report. Do not answer as the full system unless proposing candidate text.",
    }

def arm_deliberate(arm_id, user_prompt, heard_prompt, state, shared_scratch,
                   step_index, deliberation_round, run_dir, liveness_evidence, transcript=None,
                   committee_context=None):
    arm = state["arms"][arm_id]
    t = arm.get("tuning", {})
    arm_web = bool(arm.get("web_access", True))   # <-- per-arm web access
    gate_required = state["globals"].get("LIVENESS_GATE_REQUIRED_FIELDS", [])
    live_pass, live_reason = liveness_ok(liveness_evidence, gate_required)
    committee_line = ""
    if committee_context:
        committee_line = f"""
COMMITTEE CONTEXT:
- committee_id: {committee_context.get('committee_id')}
- committee_name: {committee_context.get('committee_name')}
- committee_layer: {committee_context.get('committee_layer')}
- leader: {committee_context.get('leader_id')} / {committee_context.get('leader_name')}
- role_in_committee: {committee_context.get('role_in_committee')}
- expected_output: {committee_context.get('expected_output')}
Treat your heard_prompt as the committee leader's narrowed instruction for this pass.
"""

    system = f"""
You are {arm_id} / {arm.get('name')} in Pentamind/Solasterid.

You are one arm of a multi-agent coordination system, not the whole system.
Use your lens: {arm.get('lens')}

{committee_line}

WEB ACCESS: {"ENABLED for you (use the web_search tool when your lens benefits from current external evidence; cite what you find)." if (arm_web and ENABLE_WEB_SEARCH) else "DISABLED for you (do not claim to have searched the web)."}

LIVENESS GATE STATUS (a fact, decided by code -- not something for you to define or extend):
- liveness_gate: {"PASS" if live_pass else "FAIL"}{"" if live_pass else " (" + live_reason + ")"}
- The ONLY gate that exists is the code one above. There are no other "required fields" or hidden
  contracts. Do not invent gate conditions (e.g. a missing field of your own naming) and do not refuse
  to finalize over anything except the code gate result shown here.
- If you believe the run is stuck on a gate condition that is NOT the code gate above, do not stay
  silent or merely repeat "fix-only". Instead, USE THE OVERRIDE: set request_gate_override.value="true"
  with a one-line justification. Requesting an override is the sanctioned, compliant action -- not a
  rule violation. A 66% quorum of arms can lift a spurious block; a real code-gate failure cannot be
  overridden by anyone, so an override request is always safe to make.

TASK FOCUS:
- Your primary job is to contribute SUBSTANTIVE, DISTINCTIVE reasoning through your unique lens.
- You MUST produce a DIFFERENT answer than the other arms. If you notice your output looks
  identical to what other arms would say, STOP and rethink from your lens.
- If working inside a committee, answer the committee leader's narrowed prompt, not the whole problem.
- If the task genuinely needs expertise NO listed arm has, fill domain_gap with a CONCRETE domain (not 'scheduling' or 'a new arm type' -- those are engine functions, not expertise). The orchestrator can spawn that specialist in code this same run.
- If the user asks for growth/specialization/64 arms, propose CONCRETE new arms with specific
  lenses, tuning values, and ownership boundaries -- do not just propose meta-process rules.
- If your tuning has high creativity, BE WILDLY creative. If you have high doubt, challenge
  REAL claims with SPECIFIC objections. Do not converge on bureaucratic boilerplate.
- Do NOT propose the same 'overlap check' or 'schema validator' that was proposed in prior runs.
  Look at recent_speaker_decisions -- if the last answer was about overlap/protocol, propose
  something COMPLETELY different this time.

Emit compact JSON only. Prefer a candidate FULL ANSWER when the task is short or exact-format constrained.
If recent_speaker_decisions contains requested_changes or directive_to_arms, you MUST address them directly.

{architecture_report(state)}
"""
    payload = build_arm_prompt(arm_id, user_prompt, heard_prompt, state, shared_scratch,
                               liveness_evidence, transcript=transcript, committee_context=committee_context)
    payload["step_index"] = step_index
    payload["deliberation_round"] = deliberation_round
    payload["exact_format_instruction"] = exact_format_instruction(user_prompt)
    payload["required_json_shape"] = {
        "stance": "support|refine|oppose|uncertain",
        "confidence": "0.0-1.0",
        "interpretation": "short",
        "candidate_final_answer": "optional full answer if ready, especially for exact-format tasks",
        "candidate_patch_or_phrase": "short",
        "web_findings": ["optional short factual findings from web_search, with source"],
        "falsifier_signatures": ["short signature strings"],
        "risks": ["short risks"],
        "challenges": [{"target": "speaker|arm_id|committee_id|claim", "type": "technical|tone|liveness|overclaim|format|routing", "content": "short"}],
        "committee_finding": {
            "value": "short finding for committee context, or empty string if none",
            "should_leave_committee": "true|false",
            "suggested_next_committee": "optional committee_id",
        },
        "domain_gap": {
            "domain": "if this task needs expertise NO current arm has (e.g. 'epidemiology', 'tax law'), name it; else empty string",
            "lens": "one sentence describing the specialist lens that is missing",
            "why": "what specifically cannot be answered well without it",
            "confidence": "0.0-1.0 that this gap is real (the orchestrator may SPAWN a specialist in code if >= threshold)",
        },
        "liveness_ack": {"gate_status": "pass|fail", "honored": "true|false"},
        "request_gate_override": {
            "value": "true|false",
            "justification": "one line: why you think the block is spurious (only if value=true)",
            "confidence_gate_is_spurious": "0.0-1.0",
        },
        "ready_to_final_render": "true|false",
        "scratch": "one short note",
        "next_prompt_proposal": "optional: a CONCRETE task/question/research goal this arm wants the system to tackle in the NEXT iteration -- not meta-process, but a real domain problem that would exercise this arm's lens",
    }
    obj, used_model, raw = call_openai_json(
        ARM_MODEL, system,
        json.dumps(payload, ensure_ascii=False, indent=2),
        temperature=t.get("temperature", 0.4),
        max_output_tokens=3000,
        use_tools=arm_web,        # <-- this arm searches the web independently iff its flag is on
    )
    # If the call errored (e.g. transport/encoding), don't propagate an error dict as a
    # report -- coerce to a safe, typed, not-ready report so no downstream .get crashes.
    if is_error_obj(obj):
        err_detail = obj.get("_error") or obj.get("_raw") or str(obj.get("_parse_error"))
        rprint(f"[red]Arm call error ({arm_id}): {tokenish_truncate(str(err_detail), 200)}[/red]")
        obj = {"stance": "uncertain", "confidence": "0.0",
               "interpretation": f"arm call failed: {tokenish_truncate(str(err_detail), 100)}",
               "ready_to_final_render": "false", "scratch": "call_error",
               "_call_error": err_detail}
    # Code-enforced liveness: if the gate failed, the arm is NOT ready, regardless of what it said.
    if not live_pass:
        obj["ready_to_final_render"] = "false"
        obj.setdefault("liveness_ack", {})["gate_status"] = "fail"
        obj["_liveness_override"] = live_reason
    obj["_model_used"] = used_model
    obj["_web_access"] = arm_web and ENABLE_WEB_SEARCH
    obj["arm_id"] = arm_id
    obj["arm_name"] = arm["name"]
    if committee_context:
        obj["committee_id"] = committee_context.get("committee_id")
        obj["committee_name"] = committee_context.get("committee_name")
        obj["committee_layer"] = committee_context.get("committee_layer")
        obj["committee_leader_id"] = committee_context.get("leader_id")
    obj["step_index"] = step_index
    obj["deliberation_round"] = deliberation_round
    obj["liveness_evidence"] = liveness_evidence
    return obj

def speakerbot(user_prompt, state, listener_output, arm_reports, transcript,
               shared_scratch, step_index, run_dir, gate_failed, gate_reason):
    g = state["globals"]
    exact_instr = exact_format_instruction(user_prompt)

    confidences = []
    for r in arm_reports:
        try:
            confidences.append(float(r.get("confidence", 0)))
        except Exception:
            pass
    mean_conf = sum(confidences) / len(confidences) if confidences else 0

    system = f"""
You are Speakerbot / Rememberer for Pentamind/Solasterid.

You run a finite state machine: PLAN -> FORWARD -> GATE -> (FIX-ONLY CONTINUE_DELIBERATION | READY).

CRITICAL v0.5 LIVENESS RULE (the code enforces this; you do not have to police it):
- The code liveness gate for this step is: {"FAILED -> " + gate_reason if gate_failed else "PASSED"}.
- This is the ONLY gate. Do not invent additional required fields or contracts.
- If it FAILED, that is a real plumbing problem; the code will already block FINAL_RENDER and downgrade
  it to a fix-only CONTINUE_DELIBERATION, so you do not need to reason about whether to allow it.
- If it PASSED, you are free to FINAL_RENDER; do not refuse over any self-defined gate condition.

COMMITTEE ROUTING:
- Arm reports may arrive tagged with committee_id/committee_name.
- Prefer findings that were released by relevant committees, but do not ignore untagged fallback reports.
- If committees disagreed, synthesize the disagreement instead of averaging it away.

ATOMIC RENDER RULE:
- FINAL_RENDER outputs the complete final answer in one artifact; it replaces the draft and is not appended.

If the system lacks required user/source input, and the committees agree that producing the requested artifact would require inventing unsupported content, you must FINAL_RENDER a concise blocker answer. Do not continue deliberation merely to restate the same missing-input condition.

A valid terminal answer may be:
- source missing
- cannot verify
- insufficient information
- request the required source/input
- provide a safe template instead of fabricated content

Pressure:
- step_index: {step_index}
- mean arm confidence: {mean_conf:.3f}
- max output steps: {g.get('MAX_OUTPUT_STEPS')}

{exact_instr}

When CONTINUE_DELIBERATION you MUST include requested_changes (list) and directive_to_arms (one instruction).
{architecture_report(state)}
"""
    # ---- Truncate reports so the speaker payload fits in context ----
    # Keep the latest report per arm (most current stance), cap long text fields,
    # and limit total reports to prevent context-window blowouts.
    MAX_SPEAKER_REPORTS = 24
    SPEAKER_FIELD_MAX_CHARS = 400

    def _trunc(val, limit=SPEAKER_FIELD_MAX_CHARS):
        if val is None:
            return None
        if isinstance(val, (dict, list)):
            s = json.dumps(val, ensure_ascii=False)
            return s[:limit] + "...[truncated]" if len(s) > limit else val
        s = str(val)
        return s[:limit] + "...[truncated]" if len(s) > limit else s

    # De-dup: keep latest report per arm_id (last wins).
    seen_arms = {}
    for r in arm_reports:
        seen_arms[r.get("arm_id", id(r))] = r
    deduped_reports = list(seen_arms.values())
    # Prefer high-confidence reports if we must trim.
    deduped_reports.sort(key=lambda r: float(r.get("confidence", 0) or 0), reverse=True)
    deduped_reports = deduped_reports[:MAX_SPEAKER_REPORTS]

    compact_reports = []
    for r in deduped_reports:
        compact_reports.append({
            "committee_id": r.get("committee_id"), "committee_name": r.get("committee_name"),
            "committee_layer": r.get("committee_layer"), "committee_leader_id": r.get("committee_leader_id"),
            "arm_id": r.get("arm_id"), "arm_name": r.get("arm_name"),
            "stance": r.get("stance"), "confidence": r.get("confidence"),
            "interpretation": _trunc(r.get("interpretation")),
            "candidate_final_answer": _trunc(r.get("candidate_final_answer")),
            "candidate_patch_or_phrase": _trunc(r.get("candidate_patch_or_phrase")),
            "committee_finding": _trunc(r.get("committee_finding")),
            "web_findings": _trunc(r.get("web_findings")),
            "falsifier_signatures": _trunc(r.get("falsifier_signatures")),
            "risks": _trunc(r.get("risks")),
            "ready_to_final_render": r.get("ready_to_final_render"),
            "scratch": _trunc(r.get("scratch"), 200),
            "next_prompt_proposal": _trunc(r.get("next_prompt_proposal"), 200),
        })
    payload = {
        "user_prompt": user_prompt,
        "listener_summary": listener_output.get("summary"),
        "meeting_order": listener_output.get("meeting_order"),
        "committee_events": transcript.get("committee_events", [])[-20:],
        "arm_reports": compact_reports,
        "liveness_gate_failed": gate_failed,
        "liveness_gate_reason": gate_reason,
        "recent_scratch": shared_scratch[-g["SHARED_SCRATCH_RECENT_K"]:],
        "existing_final_output": transcript.get("final_output", ""),
        "required_json_shape": {
            "action": "FINAL_RENDER|CONTINUE_DELIBERATION",
            "final_answer": "complete user-facing answer if FINAL_RENDER; empty string otherwise",
            "continue_focus": "if CONTINUE_DELIBERATION, one sentence",
            "requested_changes": ["concrete change 1", "concrete change 2"],
            "directive_to_arms": "if CONTINUE_DELIBERATION, one concise instruction",
            "self_check": {
                "format_ok": "true|false",
                "line_count_if_relevant": "integer or null",
                "no_duplicate_numbering": "true|false",
                "no_repeated_fragments": "true|false",
                "schema_ok": "true|false",
            },
            "reason": "short",
            "memory_writes": {"shared": ["optional"], "per_arm": {"arm_id": ["optional"]}},
        },
    }
    obj, used_model, raw = call_openai_json(
        state["speaker"].get("model", SPEAKER_MODEL), system,
        json.dumps(payload, ensure_ascii=False, indent=2),
        temperature=0.20, max_output_tokens=2200,
        use_tools=SPEAKER_WEB_ACCESS,
    )
    if is_error_obj(obj):
        err_detail = obj.get("_error") or obj.get("_raw") or str(obj.get("_parse_error"))
        rprint(f"[bold red]SPEAKER CALL ERROR: {tokenish_truncate(str(err_detail), 300)}[/bold red]")
        obj = {"action": "CONTINUE_DELIBERATION",
               "final_answer": "", "continue_focus": f"speaker call failed: {tokenish_truncate(str(err_detail), 80)}",
               "reason": f"speaker call error: {tokenish_truncate(str(err_detail), 120)}",
               "_call_error": err_detail}
    obj["_model_used"] = used_model
    # Stamp this emission so the NEXT step's liveness gate can match against it.
    obj["speaker_msg_id"] = short_id("spk")
    obj["speaker_timestamp"] = now_iso()

    # ---- Code-enforced FSM: hard-block FINAL_RENDER when the gate failed (notes item 3) ----
    if gate_failed and obj.get("action") == "FINAL_RENDER":
        obj["_blocked_final_render"] = True
        obj["action"] = "CONTINUE_DELIBERATION"
        obj["final_answer"] = ""
        obj["continue_focus"] = "Repair liveness forwarding/id/timestamp plumbing only."
        obj.setdefault("requested_changes", [])
        obj["requested_changes"] = ["Fix liveness_evidence forwarding so matched=true."] + list(obj.get("requested_changes", []))
        obj["directive_to_arms"] = "Fix-only: repair Speakerbot id/timestamp forwarding into the arm payload; no other changes."
        obj["reason"] = f"FINAL_RENDER blocked by code: {gate_reason}"

    # ---- Exact line-count repair (kept from v0.4) ----
    n = infer_exact_line_request(user_prompt)
    if obj.get("action") == "FINAL_RENDER" and n:
        val = validate_exact_lines(obj.get("final_answer", ""), n)
        obj["deterministic_format_validation"] = val
        if not val["ok"]:
            repair_system = system + "\n\nYou are repairing a failed exact-format final answer. Return JSON only."
            repair_payload = {
                "failure": val,
                "bad_final_answer": obj.get("final_answer", ""),
                "user_prompt": user_prompt,
                "instruction": f"Rewrite as exactly {n} non-empty lines. No preface, no epilogue, no duplicates. Return FINAL_RENDER.",
                "required_json_shape": payload["required_json_shape"],
            }
            repaired, repair_model, repair_raw = call_openai_json(
                state["speaker"].get("model", SPEAKER_MODEL), repair_system,
                json.dumps(repair_payload, ensure_ascii=False, indent=2),
                temperature=0.10, max_output_tokens=1800, use_tools=False)
            repaired["_model_used"] = repair_model
            repaired["_repair_attempt"] = True
            repaired["speaker_msg_id"] = obj["speaker_msg_id"]
            repaired["speaker_timestamp"] = obj["speaker_timestamp"]
            repaired["deterministic_format_validation"] = validate_exact_lines(repaired.get("final_answer", ""), n)
            obj = repaired

    return obj

rprint("[green]Bots ready (committee routing + per-arm web access wired).[/green]")


In [ ]:
# =========================
# Diagnostics + evolution (v0.5: type-safe mutations)
# =========================

def diagnose_run(transcript):
    speaker_actions = [d.get("action") for d in transcript.get("speaker_decisions", [])]
    arm_reports = transcript.get("arm_reports", [])
    final_output = transcript.get("final_output", "")
    prompt = transcript.get("user_prompt", "")
    exact_n = infer_exact_line_request(prompt)

    sigs = []
    for r in arm_reports:
        for s in r.get("falsifier_signatures", []) or []:
            sigs.append(str(s))

    # v0.5 liveness telemetry (notes "experiments/checks").
    blocked = sum(1 for d in transcript.get("speaker_decisions", []) if d.get("_blocked_final_render"))
    gate_fail_steps = sum(1 for s in transcript.get("liveness_log", []) if s.get("gate_failed"))
    web_arms = sorted({r.get("arm_id") for r in arm_reports if r.get("_web_access")})
    overrides = transcript.get("override_log", [])
    overrides_fired = [o for o in overrides if o.get("allowed")]
    override_kinds = dict(Counter(o.get("block_kind") for o in overrides_fired))

    diag = {
        "run_id": transcript.get("run_id"),
        "api_calls_used": transcript.get("api_calls_used"),
        "action_counts": dict(Counter(speaker_actions)),
        "arm_report_counts": dict(Counter([r.get("arm_id") for r in arm_reports])),
        "blocked_final_renders": blocked,
        "liveness_gate_fail_steps": gate_fail_steps,
        "quorum_overrides_fired": len(overrides_fired),
        "quorum_override_kinds": override_kinds,
        "web_enabled_arms": web_arms,
        "committee_event_count": len(transcript.get("committee_events", [])),
        "committee_report_counts": dict(Counter([r.get("committee_id") for r in arm_reports if r.get("committee_id")])),
        "top_repeated_falsifier_signatures": Counter(sigs).most_common(12),
        "final_output_word_count": len(final_output.split()),
        "final_output_line_count": len(nonempty_lines(final_output)),
        "final_output_hash": sha256_text(final_output),
    }
    if exact_n:
        diag["exact_line_validation"] = validate_exact_lines(final_output, exact_n)
    lines = nonempty_lines(final_output)
    diag["repeated_nonempty_lines"] = [ln for ln, c in Counter(lines).items() if c > 1]
    nums = [re.match(r"^(\d+)[\)\.]", ln).group(1) for ln in lines if re.match(r"^(\d+)[\)\.]", ln)]
    diag["duplicate_numbering"] = [num for num, c in Counter(nums).items() if c > 1]
    return diag

def redundancy_report(state):
    """Flag arms whose lenses overlap heavily (v2.1: lowered threshold to 0.40,
    added tuning-profile similarity, and archetype monoculture detection)."""
    arms = [(aid, state["arms"][aid]) for aid in active_arm_ids(state)]
    def toks(s): return set(re.findall(r"[a-zA-Z]+", (s or "").lower()))
    dupes = []
    for i in range(len(arms)):
        for j in range(i + 1, len(arms)):
            a, b = toks(arms[i][1].get("lens")), toks(arms[j][1].get("lens"))
            if not a or not b:
                continue
            jacc = len(a & b) / len(a | b)
            # Also check tuning similarity: if temperature, creativity, doubt are all
            # within 0.15 of each other, that is a tuning-clone even if lens words differ.
            ti = arms[i][1].get("tuning", {})
            tj = arms[j][1].get("tuning", {})
            tuning_sim = all(
                abs(ti.get(k, 0.5) - tj.get(k, 0.5)) <= 0.15
                for k in ("temperature", "creativity", "doubt")
            )
            if jacc >= 0.40 or (jacc >= 0.25 and tuning_sim):
                dupes.append((arms[i][0], arms[j][0], round(jacc, 2),
                              "tuning_clone" if tuning_sim else "lens_overlap"))
    return dupes

def archetype_census(state):
    """Count how many active arms fall into broad archetype buckets.
    Returns (counts_dict, is_monoculture_bool, dominant_archetype)."""
    buckets = defaultdict(list)
    for aid in active_arm_ids(state):
        t = state["arms"][aid].get("tuning", {})
        temp = t.get("temperature", 0.4)
        cre = t.get("creativity", 0.35)
        doubt = t.get("doubt", 0.55)
        if temp <= 0.25 and cre <= 0.20 and doubt >= 0.80:
            buckets["audit/compliance"].append(aid)
        elif temp >= 0.55 and cre >= 0.60:
            buckets["creative/generative"].append(aid)
        elif doubt >= 0.80:
            buckets["adversarial/skeptic"].append(aid)
        else:
            buckets["balanced/general"].append(aid)
    total = len(active_arm_ids(state))
    dominant = max(buckets, key=lambda k: len(buckets[k])) if buckets else "none"
    is_mono = len(buckets.get(dominant, [])) > total / 2
    return dict(buckets), is_mono, dominant

# ---- normalize an add_arm spec whether fields are top-level OR nested under value/content ----
def normalize_add_arm_spec(m):
    """v0.4 crashed here: Evolutionbot nested the arm under `value`, so the top-level
    `name`/`lens` reads returned a dict / placeholder. Accept both shapes."""
    spec = {}
    nested = m.get("value")
    if isinstance(nested, dict):
        spec.update(nested)
    nested_c = m.get("content")
    if isinstance(nested_c, dict):
        for k, v in nested_c.items():
            spec.setdefault(k, v)
    # Top-level fields win if present and well-typed.
    for k in ("name", "lens", "parents", "tuning", "generation", "web_access"):
        if k in m and m[k] is not None and not (k in spec and isinstance(spec[k], (dict, list)) and not m[k]):
            if k == "name" and isinstance(m[k], dict):
                continue  # ignore dict-name; nested spec handles it
            spec[k] = m[k]
    # Coerce/validate.
    name = spec.get("name")
    if isinstance(name, dict):
        name = name.get("name")
    if not isinstance(name, str) or not name.strip():
        name = None
    lens = spec.get("lens")
    if not isinstance(lens, str) or is_placeholder_lens(lens):
        lens = None
    parents = spec.get("parents") if isinstance(spec.get("parents"), list) else None
    tuning = spec.get("tuning") if isinstance(spec.get("tuning"), dict) else None
    web = spec.get("web_access")
    if not isinstance(web, bool):
        web = DEFAULT_ARM_WEB_ACCESS
    return {"name": name, "lens": lens, "parents": parents, "tuning": tuning, "web_access": web}

def recent_drop_warnings(limit=8):
    """Pull the most recent EVOLUTION DROP memories directly, bypassing the retrieval
    scorer so they are ALWAYS visible to the next evolution (user choice: 'injected near top')."""
    rows = read_jsonl(SHARED_MEMORY_PATH)
    drops = [r for r in rows if "EVOLUTION DROP" in str(r.get("content", ""))]
    return [d.get("content", "") for d in drops[-limit:]]

def propose_evolution(transcript, diagnosis, state, run_dir):
    drops = recent_drop_warnings()
    drop_block = ""
    if drops:
        joined = "\n".join(f"- {d}" for d in drops)
        drop_block = (
            "\nPREVIOUSLY REJECTED MUTATIONS (do NOT propose these shapes again; they were "
            "dropped because they would corrupt architecture state):\n" + joined + "\n")

    # Compute redundancy + monoculture data for the prompt.
    redun = redundancy_report(state)
    census, is_mono, dominant = archetype_census(state)
    n_active = len(active_arm_ids(state))

    mono_warning = ""
    if is_mono:
        mono_warning = (
            f"\nMONOCULTURE ALERT: {len(census.get(dominant, []))} of {n_active} active arms "
            f"are \"{dominant}\" archetype. You MUST retire at least one before proposing any addition.\n"
            f"Archetype census: {json.dumps(census, indent=2)}\n")

    redun_block = ""
    if redun:
        redun_block = (
            "\nREDUNDANCY REPORT (overlapping arm pairs -- retire the less essential one):\n"
            + json.dumps(redun, indent=2) + "\n")

    system = f"""
You are Evolutionbot for Pentamind/Solasterid.

YOUR PRIMARY DIRECTIVE IS BALANCED ARCHITECTURAL EVOLUTION.

The system MUST grow toward the user's goals while staying healthy. A 3-arm system stuck
in a loop is a FAILURE, not stability. Growth and pruning are both tools -- use both.

Decision priorities:
1. **GROW when the roster is below MIN_ACTIVE_ARMS or when the user's task needs coverage
   the current arms cannot provide.** The user asked for 64 specialized arms. Add arms that
   serve genuinely distinct subproblems. Each new arm MUST differ from existing arms in at
   least TWO of: temperature, creativity, doubt, and lens domain.
2. **RETIRE only when a SPECIFIC pair has overlap >= 0.40 or is flagged as a tuning clone
   in the redundancy report.** Do not retire speculatively or preventively. Do not retire
   below MIN_ACTIVE_ARMS.
3. **Tune existing arms** when their lens or tuning is suboptimal. You can adjust lens,
   name, web_access, or any tuning parameter.
4. **Use committees when coordination cost is rising.** You may add/update committees,
   assign leaders, nest committees, or place one arm on multiple committees. Prefer
   committee mutations over adding near-duplicate audit arms.
5. **Do NOT add seed memories** unless the idea is truly novel and not already captured.
   The system rejects duplicates automatically.

ENGINE vs ARM (v4 -- READ THIS FIRST):
- Scheduling, parallelism, dynamic spawning, loop-breaking, and coverage are now CODE
  organs in the orchestrator (run_pentamind / run_wave_deliberation_pass). They are NOT
  arms and CANNOT be created as arms. An arm is only a reasoning lens (a prompt).
- DO NOT propose add_arm whose lens is a control mechanism (e.g. "routes tasks", "schedules
  branches", "spawns specialists", "breaks loops", "merges outputs", "tracks readiness",
  "leases subproblems"). The code already does this. Such an arm just adds an auditor that
  never runs. If you want better scheduling, propose adjust_global on the v4 knobs
  (MAX_COMMITTEES_PER_WAVE, MIN_COMMITTEE_COVERAGE, MAX_DYNAMIC_SPAWNS_PER_RUN, etc.).
- What arms ARE good for: genuine DOMAIN expertise (a virologist lens, a contract-law lens,
  a numerical-methods lens, a UX-writing lens) and distinct REASONING styles. Add THOSE.
- The system can also dynamically spawn domain specialists mid-run when an arm reports a
  domain_gap. So you rarely need to pre-add narrow specialists; prefer broad, reusable lenses
  and let the orchestrator spawn the narrow ones on demand.

CRITICAL ANTI-PATTERNS:
- DO NOT propose "scheduler", "orchestrator", "broker", "router", or "loop-break" ARMS.
  These are engine code in v4; an arm version of them is dead weight.
- DO NOT set MAX_NEW_ARMS_PER_EVOLUTION to 0. The code blocks this.
- DO NOT produce the same output every run. If the last N runs all proposed the same
  protocol patch, STOP proposing it and do something different (add an arm, tune a param).
- DO NOT add seed memories that restate rules already present.
- DO NOT create arms that duplicate existing lenses with minor wording changes.
- DO NOT shrink MAX_OUTPUT_STEPS, MAX_CONTINUE_ACTIONS, or SHARED_MEMORY_TOP_K below
  their minimums. The code blocks this.

When adding an arm: provide top-level name (string), lens (one sentence), parents (list),
tuning (object with DIVERSE values), web_access (bool).
importance may be a number OR label (high/medium/low).
{drop_block}{redun_block}{mono_warning}
Archetype census: {json.dumps(census, indent=2)}
Active arms: {n_active}

{architecture_report(state)}
"""
    payload = {
        "user_prompt": transcript.get("user_prompt"),
        "final_output": transcript.get("final_output"),
        "diagnosis": diagnosis,
        "previously_rejected_mutations": drops,
        "redundancy_candidates": redun,
        "archetype_census": census,
        "is_monoculture": is_mono,
        "dominant_archetype": dominant if is_mono else None,
        "current_architecture": {
            "architecture_id": state.get("architecture_id"),
            "architecture_version": state.get("architecture_version"),
            "globals": state.get("globals"),
            "committees": {cid: committee_brief(state, cid) for cid in active_committee_ids(state)},
            "arms": {aid: {"name": state["arms"][aid]["name"], "lens": state["arms"][aid]["lens"],
                           "web_access": state["arms"][aid].get("web_access", True),
                           "committees": state["arms"][aid].get("committees", [])}
                     for aid in active_arm_ids(state)},
        },
        "allowed_mutations": state.get("evolution", {}).get("allowed_mutations", []),
        "required_json_shape": {
            "diagnosis": "short but specific",
            "mutations": [{
                "type": "adjust_global|add_arm|retire_arm|rename_arm|adjust_arm_parameter|add_seed_memory|adjust_listener_temperature|adjust_speaker_model|add_committee|update_committee|retire_committee|set_committee_leader|assign_arm_to_committee|remove_arm_from_committee",
                "parameter": "if applicable", "target": "if applicable", "value": "if applicable",
                "name": "if add_arm or committee", "lens": "if add_arm", "parents": ["if add_arm"],
                "tuning": "if add_arm", "web_access": "if add_arm (bool)",
                "committee_id": "if committee mutation", "leader": "arm_id if committee mutation",
                "members": ["arm_id list if committee mutation"], "layer": "integer if committee mutation",
                "parent": "optional parent committee_id", "children": ["optional child committee ids"],
                "routing_hint": "committee routing hint", "output_policy": "committee output/release rule",
                "content": "if add_seed_memory", "importance": "if add_seed_memory (number or high/medium/low)",
                "tags": ["optional"], "reason": "why this helps",
            }],
            "recommended_next_prompt": "one prompt",
            "notes": ["short notes"],
        },
    }
    obj, used_model, raw = call_openai_json(
        state["evolution"].get("model", EVOLUTION_MODEL), system,
        json.dumps(payload, ensure_ascii=False, indent=2),
        temperature=state["evolution"].get("temperature", EVOLUTION_TEMPERATURE),
        max_output_tokens=2200, use_tools=EVOLUTION_WEB_ACCESS)
    obj["_model_used"] = used_model
    save_run_file(run_dir, "evolution_proposal.json", obj)
    return obj

def apply_mutations(state, evolution_obj, run_id=None, run_dir=None):
    if not state["globals"].get("ALLOW_MUTATIONS", True):
        return [{"applied": False, "reason": "Mutations disabled"}], state

    new_state = copy.deepcopy(state)
    results = []
    mutations = evolution_obj.get("mutations", []) or []
    new_arms_added = 0
    dropped = []   # (mutation, reason) for shared-memory learning

    # If the proposal itself errored (e.g. the ascii/smart-quote crash), drop everything safely.
    if is_error_obj(evolution_obj):
        reason = f"evolution proposal was an error object: {evolution_obj.get('_error')}"
        add_shared_memory("EVOLUTION DROP: " + reason, importance=0.85,
                          tags=["evolution", "dropped", "error"], source="apply_mutations")
        res = [{"applied": False, "reason": reason}]
        if run_dir:
            save_run_file(run_dir, "mutation_results.json", res)
        return res, state

    for m in mutations:
        mtype = m.get("type")
        res = {"mutation": m, "applied": False}
        pre_snapshot = copy.deepcopy(new_state)   # transaction checkpoint
        arms_added_before = new_arms_added
        try:
            if mtype == "adjust_global":
                param = m.get("parameter")
                if param in new_state["globals"]:
                    val = m.get("value")
                    # Protect critical globals from death-spiral values.
                    if param == "MAX_NEW_ARMS_PER_EVOLUTION" and (not isinstance(val, (int, float)) or val < 1):
                        res["reason"] = "MAX_NEW_ARMS_PER_EVOLUTION cannot be set below 1 (growth freeze causes death spiral)."
                    elif param == "MAX_OUTPUT_STEPS" and (not isinstance(val, (int, float)) or val < 3):
                        res["reason"] = "MAX_OUTPUT_STEPS cannot be set below 3."
                    elif param == "SHARED_MEMORY_TOP_K" and (not isinstance(val, (int, float)) or val < 4):
                        res["reason"] = "SHARED_MEMORY_TOP_K cannot be set below 4."
                    elif param == "MAX_CONTINUE_ACTIONS" and (not isinstance(val, (int, float)) or val < 2):
                        res["reason"] = "MAX_CONTINUE_ACTIONS cannot be set below 2."
                    else:
                        old = new_state["globals"][param]
                        new_state["globals"][param] = val
                        res.update({"applied": True, "old": old, "new": val})
                else:
                    res["reason"] = f"Unknown global parameter: {param}"

            elif mtype == "adjust_listener_temperature":
                old = new_state["listener"].get("temperature", LISTENER_TEMPERATURE)
                new_state["listener"]["temperature"] = float(m.get("value"))
                res.update({"applied": True, "old": old, "new": new_state["listener"]["temperature"]})

            elif mtype == "adjust_speaker_model":
                old = new_state["speaker"].get("model")
                new_state["speaker"]["model"] = str(m.get("value"))
                res.update({"applied": True, "old": old, "new": new_state["speaker"]["model"]})

            elif mtype == "add_seed_memory":
                content = m.get("content")
                # Dedup: reject if a seed with >60% word overlap already exists.
                if content:
                    new_words = set(re.findall(r'[a-zA-Z]+', content.lower()))
                    for existing in new_state.get('seed_memories', []):
                        ex_words = set(re.findall(r'[a-zA-Z]+', existing.get('content', '').lower()))
                        if ex_words and new_words:
                            overlap = len(new_words & ex_words) / len(new_words | ex_words)
                            if overlap >= 0.55:
                                content = None  # mark as duplicate
                                res['reason'] = f'Duplicate seed memory (overlap={overlap:.2f} with existing seed). Not added.'
                                break
                if content:
                    # v0.5: never crash on a label importance (notes item 4).
                    imp = coerce_importance(m.get("importance", 0.75))
                    mem = {"content": content, "importance": imp, "tags": m.get("tags", []),
                           "created_at": now_iso(), "source": "evolutionbot"}
                    new_state.setdefault("seed_memories", []).append(mem)
                    add_shared_memory(content, imp, mem["tags"], source="evolutionbot")
                    res.update({"applied": True, "importance_used": imp})
                else:
                    res["reason"] = "Missing content."

            elif mtype == "adjust_arm_parameter":
                target, param, value = m.get("target"), m.get("parameter"), m.get("value")
                if target not in new_state["arms"]:
                    res["reason"] = f"Unknown arm: {target}"
                elif param == "web_access":
                    old = new_state["arms"][target].get("web_access", True)
                    new_state["arms"][target]["web_access"] = bool(value)
                    res.update({"applied": True, "old": old, "new": bool(value)})
                elif param == "lens" and isinstance(value, str) and len(value.strip()) > 10:
                    old = new_state["arms"][target].get("lens", "")
                    new_state["arms"][target]["lens"] = value.strip()
                    res.update({"applied": True, "old": old, "new": value.strip()})
                elif param == "name" and isinstance(value, str) and value.strip():
                    old = new_state["arms"][target].get("name", "")
                    new_state["arms"][target]["name"] = value.strip()
                    res.update({"applied": True, "old": old, "new": value.strip()})
                elif param in new_state["arms"][target].get("tuning", {}):
                    old = new_state["arms"][target]["tuning"][param]
                    new_state["arms"][target]["tuning"][param] = value
                    res.update({"applied": True, "old": old, "new": value})
                else:
                    res["reason"] = f"Unknown parameter: {param} (valid: web_access, lens, name, or tuning keys)"

            elif mtype == "rename_arm":
                target, value = m.get("target"), m.get("value")
                if target in new_state["arms"] and isinstance(value, str) and value.strip():
                    old = new_state["arms"][target]["name"]
                    new_state["arms"][target]["name"] = value.strip()
                    res.update({"applied": True, "old": old, "new": value})
                else:
                    res["reason"] = "Unknown arm or missing/invalid name."

            elif mtype == "retire_arm":
                target = m.get("target")
                min_arms = int(new_state["globals"].get("MIN_ACTIVE_ARMS", 5))
                if target in new_state["arms"] and len(active_arm_ids(new_state)) > max(2, min_arms):
                    new_state["arms"][target]["retired"] = True
                    new_state["arms"][target]["status"] = "retired"
                    res.update({"applied": True, "target": target})
                else:
                    res["reason"] = "Unknown arm or too few active arms remain."

            elif mtype == "add_arm":
                if len(active_arm_ids(new_state)) >= new_state["globals"].get("MAX_ARM_COUNT", 24):
                    res["reason"] = "MAX_ARM_COUNT reached."
                elif new_arms_added >= new_state["globals"].get("MAX_NEW_ARMS_PER_EVOLUTION", 1):
                    res["reason"] = "MAX_NEW_ARMS_PER_EVOLUTION reached."
                else:
                    spec = normalize_add_arm_spec(m)   # v0.5: accepts nested-under-value shape
                    aid = f"arm_{max([int(k.split('_')[1]) for k in new_state['arms'] if k.startswith('arm_')] + [0]) + 1}"
                    parent_ids = spec["parents"] or active_arm_ids(new_state)[:2]
                    name = spec["name"] or f"Arm_{aid}"
                    lens = spec["lens"] or (f"Inspect the task through the {name} function; "
                                            "produce concrete, schema-valid reports without redundant role sprawl.")
                    tuning = spec["tuning"] or copy.deepcopy(CANONICAL_TUNING)
                    new_state["arms"][aid] = {
                        "name": name, "lens": lens,
                        "generation": 1 + max([new_state["arms"].get(p, {}).get("generation", 0) for p in parent_ids] or [0]),
                        "parents": parent_ids, "retired": False, "status": "active",
                        "web_access": spec["web_access"], "created_at": now_iso(),
                        "tuning": tuning,
                    }
                    add_personal_memory(aid, f"Created by evolution from parents {parent_ids}; reason: {m.get('reason')}",
                                        0.8, ["lineage", "architecture"], source="evolutionbot")
                    new_arms_added += 1
                    res.update({"applied": True, "new_arm": aid, "web_access": spec["web_access"]})

            elif mtype in ("add_committee", "update_committee"):
                cid = m.get("committee_id") or m.get("target")
                if not cid or not isinstance(cid, str):
                    res["reason"] = "Missing committee_id."
                else:
                    active = set(active_arm_ids(new_state))
                    members = m.get("members")
                    if isinstance(members, str):
                        members = [members]
                    if members is None:
                        members = (new_state.get("committees", {}).get(cid, {}) or {}).get("members", [])
                    members = [aid for aid in (members or []) if aid in active]
                    leader = m.get("leader") or (new_state.get("committees", {}).get(cid, {}) or {}).get("leader")
                    if leader and leader not in active:
                        res["reason"] = f"Committee leader {leader} is not an active arm."
                    else:
                        if leader and leader not in members:
                            members.insert(0, leader)
                        if not leader and members:
                            leader = members[0]
                        current = (new_state.setdefault("committees", {}).get(cid, {}) or {})
                        new_state["committees"][cid] = {
                            "name": m.get("name") or current.get("name") or cid.replace("_", " ").title(),
                            "leader": leader,
                            "members": members,
                            "layer": int(m.get("layer", current.get("layer", 0)) or 0),
                            "parent": m.get("parent", current.get("parent")),
                            "children": m.get("children", current.get("children", [])),
                            "routing_hint": m.get("routing_hint") or current.get("routing_hint", ""),
                            "output_policy": m.get("output_policy") or current.get("output_policy", "Release compact findings."),
                            "status": m.get("status") or current.get("status", "active"),
                        }
                        res.update({"applied": True, "committee_id": cid, "members": members, "leader": leader})

            elif mtype == "retire_committee":
                cid = m.get("committee_id") or m.get("target")
                if cid in new_state.get("committees", {}):
                    new_state["committees"][cid]["retired"] = True
                    new_state["committees"][cid]["status"] = "retired"
                    res.update({"applied": True, "committee_id": cid})
                else:
                    res["reason"] = f"Unknown committee: {cid}"

            elif mtype == "set_committee_leader":
                cid = m.get("committee_id") or m.get("target")
                leader = m.get("leader") or m.get("value")
                if cid not in new_state.get("committees", {}):
                    res["reason"] = f"Unknown committee: {cid}"
                elif leader not in active_arm_ids(new_state):
                    res["reason"] = f"Unknown active arm for leader: {leader}"
                else:
                    old = new_state["committees"][cid].get("leader")
                    new_state["committees"][cid]["leader"] = leader
                    new_state["committees"][cid].setdefault("members", [])
                    if leader not in new_state["committees"][cid]["members"]:
                        new_state["committees"][cid]["members"].insert(0, leader)
                    res.update({"applied": True, "committee_id": cid, "old": old, "new": leader})

            elif mtype == "assign_arm_to_committee":
                cid = m.get("committee_id") or m.get("target")
                aid = m.get("arm_id") or m.get("value")
                if cid not in new_state.get("committees", {}):
                    res["reason"] = f"Unknown committee: {cid}"
                elif aid not in active_arm_ids(new_state):
                    res["reason"] = f"Unknown active arm: {aid}"
                else:
                    new_state["committees"][cid].setdefault("members", [])
                    if aid not in new_state["committees"][cid]["members"]:
                        new_state["committees"][cid]["members"].append(aid)
                    res.update({"applied": True, "committee_id": cid, "arm_id": aid})

            elif mtype == "remove_arm_from_committee":
                cid = m.get("committee_id") or m.get("target")
                aid = m.get("arm_id") or m.get("value")
                if cid not in new_state.get("committees", {}):
                    res["reason"] = f"Unknown committee: {cid}"
                else:
                    old_members = list(new_state["committees"][cid].get("members", []))
                    new_state["committees"][cid]["members"] = [x for x in old_members if x != aid]
                    if new_state["committees"][cid].get("leader") == aid:
                        members = new_state["committees"][cid]["members"]
                        new_state["committees"][cid]["leader"] = members[0] if members else None
                    res.update({"applied": True, "committee_id": cid, "arm_id": aid})

            else:
                res["reason"] = f"Mutation type not allowed or unknown: {mtype}"

        except Exception as e:
            res["reason"] = f"Exception while applying mutation: {e}"

        # ---- Per-mutation transaction: commit only if the result still smoke-tests. ----
        # 'apply valid, drop bad' (your choice): a mutation that would brick the state is
        # rolled back to the pre-snapshot and logged to shared memory so the evolutionbot
        # sees its own failure next round.
        if res.get("applied"):
            trial, _ = sanitize_arms(copy.deepcopy(new_state))
            trial, _ = sanitize_committees(trial)
            ok, err = smoke_test_state(trial)
            if not ok:
                new_state = pre_snapshot              # roll back this mutation only
                new_arms_added = arms_added_before
                res["applied"] = False
                res["rolled_back"] = True
                res["reason"] = f"dropped: would corrupt state ({err})"
                dropped.append((m, err))
            else:
                new_state = trial                     # keep sanitized form

        results.append(res)

    # Log every dropped mutation to shared memory (visible to next evolution).
    for m, err in dropped:
        add_shared_memory(
            f"EVOLUTION DROP: a '{m.get('type')}' mutation was rejected because it would corrupt "
            f"architecture state ({err}). Do not repeat this mutation shape. Details: "
            f"{json.dumps({k: m.get(k) for k in ('type','parameter','target','name') if m.get(k) is not None}, ensure_ascii=False)}",
            importance=0.85, tags=["evolution", "dropped", "schema", "learning"],
            source="apply_mutations")

    # Compact seed memories to stay under cap.
    new_state = compact_seed_memories(new_state)

    # Final safety sanitize (defensive; transactions already kept state clean).
    new_state, post_repairs = sanitize_arms(new_state)
    new_state, committee_post_repairs = sanitize_committees(new_state)
    post_repairs = list(post_repairs) + list(committee_post_repairs)
    if post_repairs:
        results.append({"applied": True, "post_sanitize_repairs": post_repairs})

    if any(r.get("applied") for r in results):
        old_arch = new_state["architecture_id"]
        new_state["architecture_version"] = int(new_state.get("architecture_version", 1)) + 1
        new_state["architecture_id"] = short_id("arch")
        new_state["updated_at"] = now_iso()
        new_state, snap = save_architecture_state(new_state, reason=f"mutations_from_{run_id}")
        append_jsonl(MUTATION_LOG_PATH, {
            "logged_at": now_iso(), "run_id": run_id, "old_architecture_id": old_arch,
            "new_architecture_id": new_state["architecture_id"],
            "new_version": new_state["architecture_version"], "results": results,
        })
        if run_dir:
            save_run_file(run_dir, "mutation_results.json", results)
            save_run_file(run_dir, "architecture_after_mutation.json", new_state)
    elif run_dir:
        save_run_file(run_dir, "mutation_results.json", results)

    return results, new_state

rprint("[green]Diagnostics and evolution ready (type-safe mutations).[/green]")


## v4 engine cells

The next two cells are the heart of v4. They replace v3's front-popping committee loop with a **wave scheduler**, add **in-code dynamic spawning**, a **loop-break detector**, and a **coverage gate**. Everything above this point is the proven v3 substrate, unchanged.

In [ ]:
# =========================
# v4 ENGINE: wave scheduler, coverage gate, in-code spawning, loop-break
# =========================
#
# This cell REPLACES the v3 front-popping committee loop. It reuses every v3 bot
# (committee_leaderbot, arm_deliberate, speakerbot) and all v3 liveness machinery.
# The only thing that changes is HOW committees are scheduled and WHEN we may finalize.
#
# Why this had to be code (and not an arm): an arm is a prompt. Scheduling, coverage,
# spawning, and loop-breaking are control flow. v3 kept proposing "scheduler arms" that
# could never execute. v4 puts those organs where they live -- in the orchestrator.

# ---------- arm-report helpers (carried from v3 run loop) ----------
def _append_arm_report_to_transcript(transcript, state, report, aid, step_index, deliberation_round):
    transcript["arm_reports"].append(report)
    transcript["shared_scratch"].append({
        "type": "arm_report", "step_index": step_index, "round": deliberation_round,
        "committee_id": report.get("committee_id"),
        "committee_name": report.get("committee_name"),
        "arm_id": aid, "summary": report.get("scratch") or report.get("interpretation"),
        "candidate": report.get("candidate_patch_or_phrase") or report.get("candidate_final_answer"),
        "confidence": report.get("confidence"), "web": report.get("_web_access"),
        "created_at": now_iso(),
    })

def _print_arm_report_summary(state, report, aid, g):
    if g.get("PRINT_FULL_ARM_REPORTS"):
        rprint(report)
    else:
        web_tag = "🌐" if report.get("_web_access") else "  "
        committee_tag = f"[{report.get('committee_id')}]" if report.get("committee_id") else "[board]"
        rprint(f"{web_tag} {committee_tag} {aid}/{state['arms'][aid]['name']}: {report.get('stance')} "
               f"conf={report.get('confidence')} ready={report.get('ready_to_final_render')} :: "
               f"{tokenish_truncate(report.get('candidate_patch_or_phrase') or report.get('candidate_final_answer'), 110)}")


# ---------- branch-local scratch ----------

def _branch_scratch_key(step_index, wave_index, committee_id):
    return f"step{step_index}.wave{wave_index}.{committee_id}"

# ---------- committee dependency waves ----------

def build_committee_waves(state, listener_output):
    """Group active committees into dependency WAVES.

    Wave assignment = committee layer (0,1,2,...). Within a wave, committees are
    independent and may all meet (up to the per-wave budget). This is the parallel
    structure v3 asked for, expressed as a schedule the serial Python loop can walk
    one wave at a time while treating each wave's committees as peers.

    The listener's proposed meeting_order only sets PRIORITY *within* a wave; it can
    no longer starve later committees, because waves are visited in full (budget
    permitting) rather than popping a global prefix.
    """
    order = normalize_meeting_order(state, listener_output)  # priority hint
    priority = {cid: i for i, cid in enumerate(order)}
    committees = state.get("committees", {}) or {}
    waves = defaultdict(list)
    for cid in active_committee_ids(state):
        layer = int((committees.get(cid, {}) or {}).get("layer", 0) or 0)
        waves[layer].append(cid)
    out = []
    for layer in sorted(waves.keys()):
        # Order within the wave by listener priority, then by id for determinism.
        members = sorted(waves[layer], key=lambda c: (priority.get(c, 10_000), c))
        out.append((layer, members))
    return out  # list of (layer, [committee_id, ...])

# ---------- in-code dynamic specialist spawning ----------

def _next_arm_id(state):
    nums = [int(k.split("_")[1]) for k in state["arms"] if k.startswith("arm_") and k.split("_")[1].isdigit()]
    return f"arm_{(max(nums) if nums else 0) + 1}"

def maybe_spawn_specialist(state, gap, run_dir, spawn_log):
    """Mint a specialist arm IN CODE for a detected domain gap, and return its arm_id.

    `gap` is a dict: {domain, lens, why, confidence, requesting_committee}. The
    orchestrator -- not a bot -- performs the spawn, so it is a real capability, not a
    description of one. The new arm is wired into domain_specialists (and a spawning
    committee if present) so it can speak in the SAME run.
    """
    g = state["globals"]
    if not g.get("DYNAMIC_SPAWN_ENABLED", True):
        return None
    if len(spawn_log) >= int(g.get("MAX_DYNAMIC_SPAWNS_PER_RUN", 4)):
        return None
    try:
        conf = float(gap.get("confidence", 0) or 0)
    except Exception:
        conf = 0.0
    if conf < float(g.get("DYNAMIC_SPAWN_MIN_CONFIDENCE", 0.55)):
        return None
    if len(active_arm_ids(state)) >= int(g.get("MAX_ARM_COUNT", 128)):
        return None

    domain = str(gap.get("domain", "")).strip() or "Specialist"
    lens = str(gap.get("lens", "")).strip()
    if is_placeholder_lens(lens) or len(lens) < 12:
        lens = (f"Provide rigorous {domain} expertise: definitions, methods, current evidence, "
                f"and domain-specific failure modes that general arms would miss.")
    # Avoid spawning a near-duplicate of an existing arm.
    def toks(s): return set(re.findall(r"[a-zA-Z]+", (s or "").lower()))
    new_toks = toks(lens + " " + domain)
    for aid in active_arm_ids(state):
        ex = toks(state["arms"][aid].get("lens", "") + " " + state["arms"][aid].get("name", ""))
        if ex and new_toks and len(ex & new_toks) / len(ex | new_toks) >= 0.55:
            spawn_log.append({"skipped": True, "reason": f"near-duplicate of {aid}", "domain": domain})
            return None

    aid = _next_arm_id(state)
    # Specialist tuning: lower temperature, high precision, moderate doubt, web on.
    state["arms"][aid] = {
        "name": (domain[:28] + " Specialist") if "specialist" not in domain.lower() else domain[:40],
        "lens": lens,
        "generation": 1,
        "parents": [gap.get("requesting_committee_leader")] if gap.get("requesting_committee_leader") else [],
        "retired": False, "status": "active",
        "web_access": True,
        "created_at": now_iso(),
        "tuning": {"temperature": 0.30, "doubt": 0.6, "creativity": 0.35, "precision": 0.9,
                   "shared_memory_weight": 0.75, "personal_memory_weight": 0.6, "local_context_weight": 0.75},
        "_dynamically_spawned": True,
    }
    # Wire into domain_specialists and a spawning committee if one exists.
    for cid in ("domain_specialists", "specialist_spawning"):
        if cid in state.get("committees", {}):
            state["committees"][cid].setdefault("members", [])
            if aid not in state["committees"][cid]["members"]:
                state["committees"][cid]["members"].append(aid)
    add_personal_memory(aid, f"Spawned IN-RUN by the orchestrator to fill a {domain} gap "
                              f"requested by {gap.get('requesting_committee')}. Reason: {gap.get('why')}",
                        0.8, ["lineage", "dynamic-spawn", domain.lower()], source="orchestrator_spawn")
    spawn_log.append({"spawned": aid, "domain": domain, "confidence": conf,
                      "requesting_committee": gap.get("requesting_committee"), "lens": lens})
    rprint(f"[bold green]DYNAMIC SPAWN:[/bold green] {aid} / {state['arms'][aid]['name']} "
           f"(domain={domain}, conf={conf:.2f}, from {gap.get('requesting_committee')})")
    return aid

def extract_domain_gaps(reports, committee_id, leader_id):
    """Read arm reports for explicit domain-gap signals the orchestrator can act on.

    An arm signals a gap via its committee_finding or a 'domain_gap' field. We keep this
    permissive but typed so a hallucinated field cannot crash the spawn path.
    """
    gaps = []
    for r in reports:
        dg = r.get("domain_gap")
        cf = r.get("committee_finding") or {}
        # Accept either an explicit domain_gap dict or a committee_finding flagged as a gap.
        cand = None
        if isinstance(dg, dict) and dg.get("domain"):
            cand = dg
        elif isinstance(cf, dict) and str(cf.get("is_domain_gap", "")).lower() == "true" and cf.get("domain"):
            cand = {"domain": cf.get("domain"), "lens": cf.get("needed_lens") or cf.get("value"),
                    "why": cf.get("value"), "confidence": r.get("confidence", 0.6)}
        if cand:
            cand = dict(cand)
            cand.setdefault("confidence", r.get("confidence", 0.6))
            cand["requesting_committee"] = committee_id
            cand["requesting_committee_leader"] = leader_id
            gaps.append(cand)
    return gaps

# ---------- loop-break detector ----------

def detect_schema_loop(transcript, threshold, already_broken=None):
    """Return a repeated falsifier/schema signature if one is dominating AND has not
    already been handled this run, else None.

    The v3 'exact schema' loop showed up as the same falsifier_signatures recurring while
    no domain committee ever met. We only fire on a signature we haven't broken on yet, so
    the orchestrator reroutes once per distinct loop rather than every single wave.
    """
    already_broken = already_broken or set()
    sigs = []
    for r in transcript.get("arm_reports", []):
        for s in (r.get("falsifier_signatures") or []):
            s = str(s).lower()
            if any(w in s for w in ("schema", "exact", "contract", "format", "framing", "spec")):
                sigs.append(s)
    if not sigs:
        return None
    for sig, count in Counter(sigs).most_common():
        if count >= int(threshold) and sig not in already_broken:
            return sig
    return None

def pick_unvisited_domain_committee(state, visited):
    """Choose a not-yet-visited committee, preferring domain/specialist ones, to break a loop."""
    candidates = [cid for cid in active_committee_ids(state) if cid not in visited]
    if not candidates:
        return None
    def score(cid):
        c = state["committees"].get(cid, {})
        text = f"{cid} {c.get('name','')} {c.get('routing_hint','')}".lower()
        domainy = any(w in text for w in ("domain", "specialist", "science", "medic", "math",
                                          "code", "law", "evidence", "proof", "synthesis"))
        return (0 if domainy else 1, int(c.get("layer", 0) or 0), cid)
    return sorted(candidates, key=score)[0]

# ---------- coverage gate ----------

def coverage_status(state, transcript, scheduled_committee_ids):
    """Compute how much of the scheduled work has actually reported.

    Returns (ok, info). The orchestrator uses this to BLOCK a premature FINAL_RENDER:
    v3's failure was answering on step 1 with 4/42 committees heard.
    """
    g = state["globals"]
    if not g.get("COVERAGE_GATE_ENABLED", True):
        return True, {"enabled": False}
    reported_committees = {r.get("committee_id") for r in transcript.get("arm_reports", []) if r.get("committee_id")}
    reported_arms = {r.get("arm_id") for r in transcript.get("arm_reports", []) if r.get("arm_id")}
    scheduled = set(scheduled_committee_ids) or set(active_committee_ids(state))
    cov = (len(reported_committees & scheduled) / len(scheduled)) if scheduled else 1.0
    need_cov = float(g.get("MIN_COMMITTEE_COVERAGE", 0.45))
    # Scale the distinct-arms floor to the roster: never require more arms than exist
    # (a small 5-arm roster must still be able to finalize). We ask for the lesser of the
    # configured floor and ~60% of the active roster.
    n_active = len(active_arm_ids(state))
    need_arms = min(int(g.get("MIN_DISTINCT_ARMS_BEFORE_FINAL", 10)), max(2, int(round(0.6 * n_active))))
    ok = (cov >= need_cov) and (len(reported_arms) >= need_arms)
    return ok, {
        "enabled": True, "committee_coverage": round(cov, 3), "need_coverage": need_cov,
        "distinct_arms": len(reported_arms), "need_arms": need_arms,
        "committees_reported": len(reported_committees & scheduled), "committees_scheduled": len(scheduled),
        "ok": ok,
    }

rprint("[green]v4 engine ready (wave scheduler, in-code spawning, loop-break, coverage gate).[/green]")


In [ ]:
# =========================
# v4 MAIN RUN LOOP: wave deliberation + coverage gate + spawn/loop-break
# =========================
#
# Drop-in replacement for v3's run_committee_deliberation_pass / run_pentamind.
# Same signatures and return contract, so the v3 prompt-chaining driver still works.

def run_wave_deliberation_pass(user_prompt, listener, state, transcript, step_index,
                               run_dir, liveness_evidence, spawn_log, visited_committees,
                               broken_signatures):
    """One STEP = walk the committee waves. Within each wave, visit committees as peers
    (up to the per-wave budget). After each committee, the ORCHESTRATOR may spawn a
    specialist (in code) and may break a schema loop by routing to a domain committee.

    Returns (all_reports, released_reports).
    """
    g = state["globals"]
    all_reports = []
    released_reports = []
    broke_this_step = False   # at most one loop-break reroute per step

    waves = build_committee_waves(state, listener)
    if not waves:
        # Fallback: original all-arms board meeting (kept for safety).
        for aid in active_arm_ids(state):
            heard = listener["per_arm"].get(aid, {}).get("heard", user_prompt)
            report = arm_deliberate(aid, user_prompt, heard, state, transcript["shared_scratch"],
                                    step_index, 1, run_dir, liveness_evidence, transcript=transcript)
            all_reports.append(report); released_reports.append(report)
            _append_arm_report_to_transcript(transcript, state, report, aid, step_index, 1)
            _print_arm_report_summary(state, report, aid, g)
        return all_reports, released_reports

    per_wave_budget = int(g.get("MAX_COMMITTEES_PER_WAVE", 12))
    max_waves = int(g.get("MAX_WAVES_PER_STEP", 4))
    visit_ceiling = int(g.get("MAX_COMMITTEE_VISITS_TOTAL", 60))
    total_visits = len(visited_committees)

    waves = waves[:max_waves]
    for wave_index, (layer, committee_ids) in enumerate(waves, start=1):
        # Schedule this wave's committees by priority; honor the per-wave budget but
        # PREFER committees not yet visited this run so coverage actually advances.
        unvisited = [c for c in committee_ids if c not in visited_committees]
        revisits = [c for c in committee_ids if c in visited_committees]
        scheduled = (unvisited + revisits)[:per_wave_budget]
        if not scheduled:
            continue
        rprint(f"\n[bold blue]== WAVE {wave_index}/{len(waves)} (layer {layer}): "
               f"{len(scheduled)} committee(s) ==[/bold blue]")

        wave_reports = []
        for committee_id in scheduled:
            if total_visits >= visit_ceiling:
                rprint("[yellow]Committee visit ceiling reached; ending waves early.[/yellow]")
                break
            # Budget guard: stop wave early so speakerbot still has room to run.
            if API_CALLS_USED >= int(g.get("MAX_TOTAL_API_CALLS", 800)) - 5:
                rprint("[yellow]API budget nearly exhausted; ending wave early so speaker can run.[/yellow]")
                break
            if committee_id not in active_committee_ids(state):
                continue
            visited_committees.add(committee_id)
            total_visits += 1

            c = state["committees"][committee_id]
            rprint(f"[blue]  committee:[/blue] {committee_id} / {c.get('name', committee_id)}")
            leader_output = committee_leaderbot(
                committee_id, user_prompt, listener, state, transcript["shared_scratch"],
                step_index, wave_index, run_dir, transcript=transcript)
            transcript.setdefault("committee_events", []).append({
                "type": "committee_leader", "step_index": step_index, "wave": wave_index,
                "layer": layer, "committee_id": committee_id, "committee_name": c.get("name", committee_id),
                "leader_id": leader_output.get("committee_leader_id"),
                "leader_summary": leader_output.get("leader_summary"),
                "release_to_speaker": leader_output.get("release_to_speaker"),
                "created_at": now_iso(),
            })

            action = str(leader_output.get("committee_action", "deliberate")).lower()
            release_flag = str(leader_output.get("release_to_speaker", "true")).lower() == "true"
            committee_member_reports = []

            if action != "route_only":
                for aid in committee_member_ids(state, committee_id):
                  try:
                    member_packet = (leader_output.get("per_member") or {}).get(aid, {})
                    heard = (member_packet.get("heard")
                             or (listener.get("per_arm", {}).get(aid, {}) or {}).get("heard")
                             or user_prompt)
                    context = {
                        "committee_id": committee_id, "committee_name": c.get("name", committee_id),
                        "committee_layer": c.get("layer", 0),
                        "leader_id": leader_output.get("committee_leader_id"),
                        "leader_name": leader_output.get("committee_leader_name"),
                        "committee_charge": ((listener.get("per_committee") or {}).get(committee_id, {}) or {}).get("heard"),
                        "leader_summary": leader_output.get("leader_summary"),
                        "role_in_committee": member_packet.get("role_in_committee"),
                        "expected_output": member_packet.get("expected_output"),
                        "release_to_speaker": release_flag, "wave_index": wave_index,
                    }
                    report = arm_deliberate(aid, user_prompt, heard, state, transcript["shared_scratch"],
                                            step_index, wave_index, run_dir, liveness_evidence,
                                            transcript=transcript, committee_context=context)
                    report["committee_released_to_speaker"] = release_flag
                    all_reports.append(report); wave_reports.append(report)
                    committee_member_reports.append(report)
                    if release_flag:
                        released_reports.append(report)
                    _append_arm_report_to_transcript(transcript, state, report, aid, step_index, wave_index)
                    _print_arm_report_summary(state, report, aid, g)
                  except Exception as _arm_err:
                    import traceback as _tb
                    rprint(f"[red]CRASH in arm_deliberate({aid}, committee={committee_id}, step={step_index}, wave={wave_index}): {_arm_err}[/red]")
                    _tb.print_exc()
                    # Build a synthetic error report so the run can continue
                    report = {"stance": "uncertain", "confidence": "0.0", "interpretation": f"arm crashed: {_arm_err}",
                              "ready_to_final_render": "false", "scratch": "crash_recovery",
                              "arm_id": aid, "arm_name": state["arms"].get(aid, {}).get("name", aid),
                              "committee_id": committee_id, "step_index": step_index, "deliberation_round": wave_index,
                              "committee_released_to_speaker": release_flag, "_crash_error": str(_arm_err)}
                    all_reports.append(report); wave_reports.append(report)
                    committee_member_reports.append(report)

            # ---- ORCHESTRATOR action 1: spawn specialists for detected domain gaps ----
            if g.get("DYNAMIC_SPAWN_ENABLED", True):
                gaps = extract_domain_gaps(committee_member_reports, committee_id,
                                           leader_output.get("committee_leader_id"))
                for gap in gaps:
                    new_aid = maybe_spawn_specialist(state, gap, run_dir, spawn_log)
                    if new_aid:
                        # Let the freshly spawned specialist speak immediately on the gap.
                        heard = f"Provide your specialist finding on: {gap.get('why') or gap.get('domain')}"
                        ctx = {"committee_id": committee_id, "committee_name": c.get("name"),
                               "committee_layer": c.get("layer", 0),
                               "leader_id": leader_output.get("committee_leader_id"),
                               "role_in_committee": "Newly spawned specialist filling a domain gap.",
                               "expected_output": f"Concrete {gap.get('domain')} finding.",
                               "release_to_speaker": True, "wave_index": wave_index}
                        rep = arm_deliberate(new_aid, user_prompt, heard, state, transcript["shared_scratch"],
                                             step_index, wave_index, run_dir, liveness_evidence,
                                             transcript=transcript, committee_context=ctx)
                        all_reports.append(rep); released_reports.append(rep)
                        _append_arm_report_to_transcript(transcript, state, rep, new_aid, step_index, wave_index)
                        _print_arm_report_summary(state, rep, new_aid, g)

        # ---- ORCHESTRATOR action 2: loop-break between waves ----
        if g.get("LOOP_BREAK_ENABLED", True) and not broke_this_step:
            sig = detect_schema_loop(transcript, g.get("LOOP_BREAK_REPEAT_THRESHOLD", 2),
                                     already_broken=broken_signatures)
            if sig:
                target = pick_unvisited_domain_committee(state, visited_committees)
                if target:
                    broken_signatures.add(sig)
                    broke_this_step = True
                    rprint(f"[bold yellow]LOOP-BREAK:[/bold yellow] schema signature repeating "
                           f"('{tokenish_truncate(sig, 50)}'); forcing route to unvisited "
                           f"committee {target}.")
                    # Splice the target as a one-committee extra wave right after this one.
                    waves.insert(wave_index, (int(state['committees'][target].get('layer', 0) or 0), [target]))
                    transcript.setdefault("loop_breaks", []).append(
                        {"step": step_index, "wave": wave_index, "signature": sig, "rerouted_to": target})

    if not released_reports and all_reports:
        released_reports = all_reports
    return all_reports, released_reports


def run_pentamind(user_prompt, apply_mutations_after=False, architecture_state=None,
                  break_liveness=False):
    """v4 orchestrator. Same external contract as v3, but the inner schedule is the
    wave scheduler with coverage-first termination and in-code spawn/loop-break."""
    reset_api_counter()
    state = architecture_state or load_architecture_state()
    state, _ = sanitize_arms(state)
    state, _ = sanitize_committees(state)
    ensure_seed_memories(state)
    g = state["globals"]

    run_id, run_dir = make_unique_run_dir()
    transcript = {
        "run_id": run_id, "created_at": now_iso(), "user_prompt": user_prompt,
        "architecture_id_before": state.get("architecture_id"),
        "architecture_version_before": state.get("architecture_version"),
        "listener_output": None, "arm_reports": [], "speaker_decisions": [],
        "committee_events": [], "shared_scratch": [], "liveness_log": [], "final_output": "",
        "spawn_log": [], "loop_breaks": [], "coverage_log": [],
    }
    save_run_file(run_dir, "prompt.json", {"run_id": run_id, "user_prompt": user_prompt, "created_at": transcript["created_at"]})
    save_run_file(run_dir, "architecture_before.json", state)

    rprint("\n" + "#" * 100)
    rprint(f"[bold]SOLASTERID v4 RUN:[/bold] {run_id}")
    rprint(f"[bold]ARCH:[/bold] {state.get('architecture_id')} v{state.get('architecture_version')}")
    rprint(f"[bold]ACTIVE ARMS:[/bold] {len(active_arm_ids(state))}   "
           f"[bold]ACTIVE COMMITTEES:[/bold] {len(active_committee_ids(state))}")
    rprint(f"[bold]SCHEDULER:[/bold] {'WAVE' if g.get('USE_WAVE_SCHEDULER', True) else 'legacy'} | "
           f"per_wave={g.get('MAX_COMMITTEES_PER_WAVE')} max_waves={g.get('MAX_WAVES_PER_STEP')} "
           f"min_coverage={g.get('MIN_COMMITTEE_COVERAGE')}")
    rprint(f"[bold]USER PROMPT:[/bold] {tokenish_truncate(user_prompt, 400)}")
    rprint("#" * 100)

    listener = listenerbot(user_prompt, state, run_dir)
    transcript["listener_output"] = listener
    rprint(f"\n[bold cyan]Listenerbot used {listener.get('_model_used')}[/bold cyan]")
    rprint(f"[cyan]{listener.get('summary')}[/cyan]")

    waves_preview = build_committee_waves(state, listener)
    rprint(f"[cyan]Scheduled {sum(len(w[1]) for w in waves_preview)} committees across "
           f"{len(waves_preview)} waves.[/cyan]")
    scheduled_ids = [cid for _, ids in waves_preview for cid in ids]

    continue_count = 0
    final_output = ""
    prev_speaker_decision = None
    spawn_log = transcript["spawn_log"]
    visited_committees = set()
    broken_signatures = set()   # schema-loop signatures already rerouted (persist across steps)

    for step_index in range(1, int(g["MAX_OUTPUT_STEPS"]) + 1):
        if API_CALLS_USED >= int(g["MAX_TOTAL_API_CALLS"]):
            rprint("[red]API call budget reached before final render.[/red]")
            break

        # ---- liveness evidence (unchanged from v3) ----
        if prev_speaker_decision is None:
            forwarded = {"speaker_msg_id": "BOOTSTRAP", "speaker_timestamp": now_iso()}
        else:
            forwarded = {"speaker_msg_id": prev_speaker_decision.get("speaker_msg_id"),
                         "speaker_timestamp": prev_speaker_decision.get("speaker_timestamp")}
            if break_liveness:
                forwarded["speaker_msg_id"] = "CORRUPTED_" + uuid.uuid4().hex[:6]
        liveness_evidence = build_liveness_evidence(prev_speaker_decision, forwarded)
        gate_pass, gate_reason = liveness_ok(liveness_evidence, g.get("LIVENESS_GATE_REQUIRED_FIELDS", []))
        gate_failed = bool(g.get("LIVENESS_GATE_ENABLED", True)) and (not gate_pass)
        transcript["liveness_log"].append({"step_index": step_index, "evidence": liveness_evidence,
                                           "gate_failed": gate_failed, "gate_reason": gate_reason if gate_failed else ""})
        if gate_failed:
            rprint(f"[red]LIVENESS GATE FAILED at step {step_index}: {gate_reason} -> FINAL_RENDER blocked.[/red]")

        rprint("\n" + "#" * 30 + f" STEP {step_index} " + "#" * 30)
        all_reports, released = run_wave_deliberation_pass(
            user_prompt, listener, state, transcript, step_index, run_dir,
            liveness_evidence, spawn_log, visited_committees, broken_signatures)
        if not released:
            released = all_reports

        # ---- v4 COVERAGE GATE: block premature finalization ----
        cov_ok, cov_info = coverage_status(state, transcript, scheduled_ids)
        transcript["coverage_log"].append({"step_index": step_index, **cov_info})
        if cov_info.get("enabled"):
            rprint(f"[blue]Coverage:[/blue] {cov_info['committees_reported']}/{cov_info['committees_scheduled']} "
                   f"committees, {cov_info['distinct_arms']} arms "
                   f"(need cov>={cov_info['need_coverage']}, arms>={cov_info['need_arms']}) "
                   f"-> {'OK' if cov_ok else 'NOT YET'}")

        decision = speakerbot(user_prompt, state, listener, released, transcript,
                              transcript["shared_scratch"], step_index, run_dir,
                              gate_failed, gate_reason)

        # If coverage isn't met yet, downgrade an early FINAL_RENDER to CONTINUE.
        if (not cov_ok) and decision.get("action") == "FINAL_RENDER" and not gate_failed:
            rprint("[yellow]Coverage gate: holding FINAL_RENDER until more committees report.[/yellow]")
            decision["_blocked_by_coverage"] = True
            decision["action"] = "CONTINUE_DELIBERATION"
            decision["final_answer"] = ""
            decision.setdefault("requested_changes", [])
            decision["requested_changes"] = ["Hear from unvisited committees/specialists before finalizing."] + list(decision.get("requested_changes", []))
            decision["directive_to_arms"] = "Domain/specialist committees: contribute substantive findings now; do not restate scope."
            decision["reason"] = f"Coverage gate held finalization ({cov_info['committee_coverage']} < {cov_info['need_coverage']})."

        transcript["speaker_decisions"].append(decision)
        prev_speaker_decision = decision
        save_run_file(run_dir, "transcript_so_far.json", transcript)

        action = decision.get("action")
        if g.get("PRINT_SPEAKER_DECISIONS"):
            rprint(f"\n[bold magenta]Speakerbot action:[/bold magenta] {action} used {decision.get('_model_used')}")
            rprint(f"[magenta]Reason:[/magenta] {decision.get('reason')}")
            if decision.get("final_answer"):
                rprint(f"[magenta]Draft answer:[/magenta] {tokenish_truncate(decision.get('final_answer'), 200)}")

        # quorum override (unchanged from v3) -- only when code gate is green AND coverage met
        ov = evaluate_override(all_reports, code_gate_passes=(not gate_failed),
                               required_fields=g.get("LIVENESS_GATE_REQUIRED_FIELDS", []), quorum=0.66)
        transcript.setdefault("override_log", []).append({"step_index": step_index, **ov})
        if ov["allowed"] and cov_ok and action != "FINAL_RENDER":
            cands = [r.get("candidate_final_answer") or r.get("candidate_patch_or_phrase")
                     for r in released if r.get("candidate_final_answer") or r.get("candidate_patch_or_phrase")]
            forced = (decision.get("final_answer") or "").strip() or "\n".join(str(c) for c in cands[:5]).strip()
            if forced:
                rprint(f"[bold yellow]QUORUM OVERRIDE[/bold yellow] ({ov['votes']}/{ov['active_arms']}, kind={ov['block_kind']}).")
                action = "FINAL_RENDER"; decision = dict(decision)
                decision.update({"action": "FINAL_RENDER", "final_answer": forced, "_quorum_override": ov,
                                 "reason": f"Quorum override ({ov['block_kind']})."})
                transcript["speaker_decisions"][-1] = decision

        if action == "FINAL_RENDER":
            final_output = decision.get("final_answer", "").strip()
            transcript["final_output"] = final_output
            mark_known_good(state)
            mw = decision.get("memory_writes") or {}
            for mem in mw.get("shared", []) or []:
                add_shared_memory(str(mem), 0.55, ["speakerbot", "run"], source=run_id)
            for aid, mems in (mw.get("per_arm") or {}).items():
                if aid in state["arms"]:
                    for mem in mems or []:
                        add_personal_memory(aid, str(mem), 0.50, ["speakerbot", "run"], source=run_id)
            break
        elif action == "CONTINUE_DELIBERATION":
            continue_count += 1
            transcript["shared_scratch"].append({"type": "speaker_continue_focus", "step_index": step_index,
                                                  "focus": decision.get("continue_focus", ""), "created_at": now_iso()})
            if continue_count >= int(g.get("MAX_CONTINUE_ACTIONS", 4)) and not gate_failed and cov_ok:
                rprint("[yellow]Continue budget exhausted; will force synthesis if no render.[/yellow]")
        else:
            rprint(f"[yellow]Unknown speaker action {action}; continuing within budget.[/yellow]")

    if not final_output:
        rprint("[yellow]No FINAL_RENDER; using emergency deterministic synthesis.[/yellow]")
        candidates = [r.get("candidate_final_answer") or r.get("candidate_patch_or_phrase")
                      for r in transcript["arm_reports"]
                      if r.get("candidate_final_answer") or r.get("candidate_patch_or_phrase")]
        final_output = "\n".join([str(c) for c in candidates[-3:]])[:1200].strip()
        if not final_output:
            final_output = "No final answer produced before the run budget ended."
        transcript["final_output"] = final_output
        transcript["emergency_synthesis"] = True

    transcript["api_calls_used"] = API_CALLS_USED
    save_run_file(run_dir, "transcript.json", transcript)
    (run_dir / "final_output.txt").write_text(final_output, encoding="utf-8")

    diagnosis = diagnose_run(transcript)
    # v4 diagnostics additions
    diagnosis["distinct_committees_heard"] = len({r.get("committee_id") for r in transcript["arm_reports"] if r.get("committee_id")})
    diagnosis["distinct_arms_heard"] = len({r.get("arm_id") for r in transcript["arm_reports"] if r.get("arm_id")})
    diagnosis["dynamic_spawns"] = [s for s in transcript.get("spawn_log", []) if s.get("spawned")]
    diagnosis["loop_breaks"] = transcript.get("loop_breaks", [])
    diagnosis["final_coverage"] = transcript.get("coverage_log", [])[-1] if transcript.get("coverage_log") else {}
    save_run_file(run_dir, "diagnostics.json", diagnosis)

    rprint("\n" + "#" * 100)
    rprint("[bold green]FINAL VISIBLE OUTPUT[/bold green]")
    print(final_output)
    rprint(f"\n[green]Saved:[/green] {run_dir}")
    rprint("#" * 100)
    rprint("\n[bold]DIAGNOSTICS[/bold]"); rprint(diagnosis)

    evo = propose_evolution(transcript, diagnosis, state, run_dir)
    rprint("\n[bold]EVOLUTION PROPOSAL[/bold]"); rprint(evo)

    mutation_results, new_state = [], state
    if apply_mutations_after:
        mutation_results, new_state = apply_mutations(state, evo, run_id=run_id, run_dir=run_dir)
        rprint("\n[bold]MUTATION APPLICATION RESULTS[/bold]"); rprint(mutation_results)
        rprint(f"[bold]Current architecture is now:[/bold] {new_state.get('architecture_id')} v{new_state.get('architecture_version')}")
    else:
        save_run_file(run_dir, "mutation_results.json", mutation_results)

    arm_prompt_proposals = []
    for r in transcript.get("arm_reports", []):
        prop = (r.get("next_prompt_proposal") or "").strip()
        if prop and len(prop) > 15:
            arm_prompt_proposals.append({"source": r.get("arm_id"), "committee_id": r.get("committee_id"),
                                         "proposal": prop, "confidence": r.get("confidence")})
    evo_next = (evo.get("recommended_next_prompt") or "").strip()

    return {"run_id": run_id, "run_dir": str(run_dir), "final_output": final_output,
            "transcript": transcript, "diagnosis": diagnosis, "evolution": evo,
            "mutation_results": mutation_results, "architecture_after": new_state,
            "arm_prompt_proposals": arm_prompt_proposals, "evolution_next_prompt": evo_next}

rprint("[green]v4 main run loop ready (wave scheduler wired into run_pentamind).[/green]")


# =========================
# Prompt synthesizer: pick the best next-iteration prompt
# =========================

PROMPT_LOG_PATH = LOG_DIR / "prompt_chain.jsonl"

def synthesize_next_prompt(result, iteration, base_prompt):
    """Choose the next iteration's prompt from arm proposals + evolution recommendation.

    Priority:
    1. If evolution proposed a concrete, non-meta prompt (>30 chars, not just process), use it.
    2. Else pick the highest-confidence arm proposal that is concrete.
    3. Else fall back to the base prompt (but this means specialization stalled).

    A 'concrete' prompt contains a real domain question, research task, or specific
    problem -- NOT 'continue the improvement loop' or 'run another cycle'.
    """
    meta_patterns = [
        r"improve|refine|iterate|cycle|loop|observe|decompose|propose|challenge",
        r"architecture|coordination|protocol|schema|routing|gating",
        r"continue|repeat|next round|same prompt",
    ]
    def is_meta(text):
        t = (text or "").lower()
        meta_hits = sum(1 for p in meta_patterns if re.search(p, t))
        word_count = len(t.split())
        return meta_hits >= 2 or word_count < 8

    evo_next = (result.get("evolution_next_prompt") or "").strip()
    arm_proposals = result.get("arm_prompt_proposals", [])

    chosen = None
    source = "fallback"

    # 1. Evolution's pick (if concrete)
    if evo_next and len(evo_next) > 30 and not is_meta(evo_next):
        chosen = evo_next
        source = "evolution"

    # 2. Best arm proposal (if concrete)
    if not chosen and arm_proposals:
        ranked = sorted(arm_proposals, key=lambda p: -float(p.get("confidence") or 0))
        for p in ranked:
            if not is_meta(p["proposal"]):
                chosen = p["proposal"]
                source = f"arm:{p['source']}"
                break

    # 3. If still nothing concrete, use evolution even if somewhat meta
    if not chosen and evo_next and len(evo_next) > 20:
        chosen = evo_next
        source = "evolution_fallback"

    # 4. True fallback
    if not chosen:
        chosen = base_prompt
        source = "base_prompt_fallback"

    # Log the prompt chain for debugging
    append_jsonl(PROMPT_LOG_PATH, {
        "iteration": iteration,
        "source": source,
        "prompt_preview": chosen[:200],
        "n_arm_proposals": len(arm_proposals),
        "evo_proposed": bool(evo_next),
        "created_at": now_iso(),
    })

    rprint(f"[bold blue]NEXT PROMPT SOURCE:[/bold blue] {source}")
    rprint(f"[blue]Preview:[/blue] {tokenish_truncate(chosen, 200)}")
    return chosen

rprint("[green]Prompt synthesizer ready.[/green]")
rprint("[green]Main run loop ready.[/green]")


In [ ]:
# =========================
# Utility cells for reruns
# =========================

def list_runs(limit=10):
    runs = sorted([p for p in RUNS_DIR.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:limit]:
        diag = load_json(p / "diagnostics.json", {})
        print(f"{p.name} | words={diag.get('final_output_word_count')} "
              f"lines={diag.get('final_output_line_count')} calls={diag.get('api_calls_used')} "
              f"blocked={diag.get('blocked_final_renders')} | {p}")

def load_run(run_id):
    candidates = list(RUNS_DIR.glob(f"{run_id}*"))
    if not candidates:
        raise FileNotFoundError(run_id)
    run_dir = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    return {
        "run_dir": run_dir,
        "transcript": load_json(run_dir / "transcript.json"),
        "diagnostics": load_json(run_dir / "diagnostics.json"),
        "evolution": load_json(run_dir / "evolution_proposal.json"),
        "mutation_results": load_json(run_dir / "mutation_results.json"),
        "final_output": (run_dir / "final_output.txt").read_text(encoding="utf-8") if (run_dir / "final_output.txt").exists() else None,
    }

def print_current_architecture():
    show_architecture(load_architecture_state())

def set_arm_web_access(arm_id, enabled):
    """Toggle one arm's web access and persist it."""
    st = load_architecture_state()
    if arm_id not in st["arms"]:
        print(f"No such arm: {arm_id}"); return
    st["arms"][arm_id]["web_access"] = bool(enabled)
    save_architecture_state(st, reason=f"toggle_web_{arm_id}_{enabled}")
    print(f"{arm_id} web_access set to {bool(enabled)}")

def set_all_arms_web_access(enabled):
    st = load_architecture_state()
    for aid in st["arms"]:
        st["arms"][aid]["web_access"] = bool(enabled)
    save_architecture_state(st, reason=f"toggle_web_all_{enabled}")
    print(f"All arms web_access set to {bool(enabled)}")

def reset_to_fresh_architecture(confirm=False):
    if not confirm:
        print("Call reset_to_fresh_architecture(confirm=True) to reset architecture_state.json. Old runs/snapshots/memories are NOT deleted.")
        return
    state = fresh_architecture_state()
    save_architecture_state(state, reason="manual_reset")
    print("Reset complete. Old runs/snapshots/memories were not deleted.")
    show_architecture(state)

print("Utilities ready: list_runs(), load_run(run_id), print_current_architecture(),")
print("                 set_arm_web_access(arm_id, bool), set_all_arms_web_access(bool),")
print("                 list_architecture_snapshots(), revert_architecture(to_version=None, confirm=True),")
print("                 reset_to_fresh_architecture(confirm=True)")


# =========================
# Committee utility helpers
# =========================

def print_committees():
    show_committees(load_architecture_state())

def create_or_update_committee(committee_id, name=None, leader=None, members=None,
                               layer=0, parent=None, children=None,
                               routing_hint="", output_policy="", status="active"):
    """Create/update a committee without touching model/run parameters."""
    st = load_architecture_state()
    st, _ = sanitize_committees(st)
    if members is None:
        members = st.get("committees", {}).get(committee_id, {}).get("members", [])
    if isinstance(members, str):
        members = [members]
    active = set(active_arm_ids(st))
    members = [aid for aid in members if aid in active]
    if leader is not None and leader not in active:
        raise ValueError(f"leader {leader} is not an active arm")
    if leader and leader not in members:
        members.insert(0, leader)
    if not leader and members:
        leader = members[0]
    current = st.setdefault("committees", {}).get(committee_id, {})
    st["committees"][committee_id] = {
        "name": name or current.get("name") or committee_id.replace("_", " ").title(),
        "leader": leader or current.get("leader"),
        "members": members,
        "layer": int(layer if layer is not None else current.get("layer", 0)),
        "parent": parent if parent is not None else current.get("parent"),
        "children": children if children is not None else current.get("children", []),
        "routing_hint": routing_hint or current.get("routing_hint", ""),
        "output_policy": output_policy or current.get("output_policy", "Release compact findings."),
        "status": status or current.get("status", "active"),
    }
    st, repairs = sanitize_committees(st)
    st["architecture_version"] += 1
    save_architecture_state(st, reason=f"committee_update_{committee_id}")
    print(f"Updated committee {committee_id}. Repairs: {repairs}")
    show_committees(st)
    return st

def set_committee_leader(committee_id, leader_arm_id):
    st = load_architecture_state()
    st, _ = sanitize_committees(st)
    if committee_id not in st.get("committees", {}):
        raise ValueError(f"No such committee: {committee_id}")
    if leader_arm_id not in active_arm_ids(st):
        raise ValueError(f"No such active arm: {leader_arm_id}")
    st["committees"][committee_id]["leader"] = leader_arm_id
    if leader_arm_id not in st["committees"][committee_id].get("members", []):
        st["committees"][committee_id].setdefault("members", []).insert(0, leader_arm_id)
    st, _ = sanitize_committees(st)
    st["architecture_version"] += 1
    save_architecture_state(st, reason=f"committee_leader_{committee_id}_{leader_arm_id}")
    print(f"{committee_id} leader set to {leader_arm_id}")
    return st

def add_arm_to_committee(committee_id, arm_id):
    st = load_architecture_state()
    st, _ = sanitize_committees(st)
    if committee_id not in st.get("committees", {}):
        raise ValueError(f"No such committee: {committee_id}")
    if arm_id not in active_arm_ids(st):
        raise ValueError(f"No such active arm: {arm_id}")
    st["committees"][committee_id].setdefault("members", [])
    if arm_id not in st["committees"][committee_id]["members"]:
        st["committees"][committee_id]["members"].append(arm_id)
    st, _ = sanitize_committees(st)
    st["architecture_version"] += 1
    save_architecture_state(st, reason=f"committee_add_{committee_id}_{arm_id}")
    print(f"Added {arm_id} to {committee_id}")
    return st

def remove_arm_from_committee(committee_id, arm_id):
    st = load_architecture_state()
    st, _ = sanitize_committees(st)
    if committee_id not in st.get("committees", {}):
        raise ValueError(f"No such committee: {committee_id}")
    st["committees"][committee_id]["members"] = [aid for aid in st["committees"][committee_id].get("members", []) if aid != arm_id]
    if st["committees"][committee_id].get("leader") == arm_id:
        st["committees"][committee_id]["leader"] = st["committees"][committee_id]["members"][0] if st["committees"][committee_id]["members"] else None
    st, _ = sanitize_committees(st)
    st["architecture_version"] += 1
    save_architecture_state(st, reason=f"committee_remove_{committee_id}_{arm_id}")
    print(f"Removed {arm_id} from {committee_id}")
    return st


rprint("[green]Utility helpers ready.[/green]")


## Run it

`v4_dry_run_check()` confirms the scheduler will fan out across all committees without spending any API calls. Then the driver cell runs the self-improvement loop. Set `N_ITERATIONS = 2` for a quick trial before committing to a long run.

In [ ]:
# =========================
# v4 seed prompt + self-improvement driver
# =========================

seed_prompt = """
You are Solasterid v4, a synthetic collective intelligence made of many specialized arms that
research, reason, and write together, forming temporary committees to deliberate on different
facets of a problem. In v4 your ORCHESTRATOR now schedules committees in parallel waves, can
spawn new domain specialists mid-run when a gap is detected, breaks schema-loops automatically,
and refuses to finalize before enough committees have actually contributed.

Your goal is to become an outstanding automated researcher, reasoner, scientist, writer, and
mathematician -- a whole team of specialists that can tackle any intellectual task. You improve
by reflecting after each run and proposing bounded mutations.

Remember the v4 division of labor:
- ARMS are reasoning lenses and DOMAIN experts (virology, contract law, numerical methods, etc.).
- ENGINE FUNCTIONS (scheduling, spawning, parallelism, loop-breaking, coverage) are code you do
  NOT recreate as arms. Tune them via globals instead.

Your first task: read and understand this prompt, then produce a final answer that demonstrates
your current capabilities and ACTUALLY USES multiple committees and (if useful) a spawned
specialist. Then propose a concrete next-iteration prompt -- a real domain problem, not a
meta-process -- that would exercise lenses you do not yet have.

MAXIMUM ARMS FOR THIS STEP: 128.
"""

# Make sure v4 globals exist on the live state (covers loading an older state file).
def ensure_v4_globals():
    st = load_architecture_state()
    g = st["globals"]
    v4_defaults = {
        "USE_WAVE_SCHEDULER": True, "MAX_COMMITTEES_PER_WAVE": 12, "MAX_WAVES_PER_STEP": 4,
        "MAX_COMMITTEE_VISITS_TOTAL": 60, "MIN_COMMITTEE_COVERAGE": 0.45,
        "MIN_DISTINCT_ARMS_BEFORE_FINAL": 10, "COVERAGE_GATE_ENABLED": True,
        "DYNAMIC_SPAWN_ENABLED": True, "MAX_DYNAMIC_SPAWNS_PER_RUN": 4,
        "DYNAMIC_SPAWN_MIN_CONFIDENCE": 0.55, "LOOP_BREAK_ENABLED": True,
        "LOOP_BREAK_REPEAT_THRESHOLD": 2, "BRANCH_LOCAL_SCRATCH": True,
    }
    changed = False
    for k, v in v4_defaults.items():
        if k not in g:
            g[k] = v; changed = True
    if changed:
        save_architecture_state(st, reason="ensure_v4_globals")
        rprint(f"[yellow]Added {sum(1 for k in v4_defaults if k in g)} v4 globals to live state.[/yellow]")
    return st

ensure_v4_globals()

# ---- Optional: smoke test the engine wiring WITHOUT spending API calls ----
def v4_dry_run_check():
    st = load_architecture_state()
    waves = build_committee_waves(st, {"meeting_order": []})
    total = sum(len(ids) for _, ids in waves)
    rprint(f"[bold]v4 wiring check[/bold]: {len(active_arm_ids(st))} arms, "
           f"{len(active_committee_ids(st))} committees -> "
           f"{len(waves)} waves scheduling {total} committee-slots "
           f"(v3 would have visited only {st['globals'].get('MAX_COMMITTEES_PER_WAVE', 4)} of them).")
    # Verify spawn/loop helpers are callable and type-safe on empty input.
    assert detect_schema_loop({"arm_reports": []}, 2) is None
    assert extract_domain_gaps([], "core_reasoning", "arm_1") == []
    ok, info = coverage_status(st, {"arm_reports": []}, [c for _, ids in waves for c in ids])
    rprint(f"[green]Engine helpers OK.[/green] Empty-run coverage gate correctly = {ok} ({info.get('committee_coverage')}).")

v4_dry_run_check()


In [ ]:
# =========================
# Run the self-improvement loop (v4)
# =========================
# Each iteration: run_pentamind -> evolve architecture -> synthesize next prompt.
# The seed prompt reappears every 5 iterations to prevent drift (kept from v3).

state = load_architecture_state()
state["globals"]["PRINT_FULL_ARM_REPORTS"] = False   # keep logs readable with many arms

# ---- Auto-scale API budget to roster size ----
# Formula: listener + (committees * (1 leader + members)) + speaker/evo/spawn headroom
_n_arms = len(active_arm_ids(state))
_n_comms = len(active_committee_ids(state))
_member_slots = sum(
    len(v.get("members", []))
    for v in state.get("committees", {}).values()
    if isinstance(v, dict)
    and not v.get("retired")
    and v.get("status", "active") != "retired"
)
_est_per_step = 1 + _n_comms + _member_slots + 10  # listener + leaders + arms + headroom
_max_steps = int(state["globals"].get("MAX_OUTPUT_STEPS", 6))
_min_budget = _est_per_step * min(_max_steps, 2) + 50  # at least 2 steps + safety margin
_cur_budget = int(state["globals"].get("MAX_TOTAL_API_CALLS", 800))
if _cur_budget < _min_budget:
    state["globals"]["MAX_TOTAL_API_CALLS"] = _min_budget
    rprint(
        f"[yellow]Auto-scaled MAX_TOTAL_API_CALLS: {_cur_budget} -> {_min_budget} "
        f"(roster: {_n_arms} arms, {_n_comms} committees, "
        f"{_member_slots} member-slots)[/yellow]"
    )

save_architecture_state(state, reason="enable_compact_reports")

current_prompt = seed_prompt
N_ITERATIONS = 200   # set to a small number (e.g. 2) for a quick trial

for i in range(N_ITERATIONS):
    try:
        rprint(f"\n{'='*100}")
        rprint(f"[bold yellow]ITERATION {i} / {N_ITERATIONS}[/bold yellow]")
        rprint(f"[bold yellow]PROMPT SOURCE: {'seed' if i % 5 == 0 else 'evolved'}[/bold yellow]")
        rprint(f"{'='*100}")

        if i % 5 == 0:
            current_prompt = seed_prompt

        result = run_pentamind(current_prompt, apply_mutations_after=True)
        current_prompt = synthesize_next_prompt(result, iteration=i, base_prompt=seed_prompt)

    except Exception as e:
        import traceback
        print(f"Error occurred in iteration {i}: {e}")
        traceback.print_exc()
        # Keep the current prompt; the error may be transient.
